<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 📑 Table of Contents: E-003 Hierarchical ICD-10 Classification

---

### 🧠 Hierarchical Two-Stage Architecture
Experiment objective, motivation from E-002 chapter confusion analysis,
and comparison baseline.

### 🧪 Experiment Log: Scientific Record (E-003)
Official configuration and results.

### 🔬 Phase 1: Experiment Configuration
MLflow SQLite backend, Gold layer Parquet discovery, E-001 registry
path verification, Stage-1 and Stage-2 hyperparameters.

### ⚙️ Phase 1b: Environment Setup & Imports
HuggingFace local cache, MPS fallback, seed locking, Stage-1/Stage-2
output directories.

### 📥 Phase 2: Data Loading & Label Derivation
Gold layer ingestion, billable-only filter (9,660 records), chapter
label derivation (22 classes), shared train/val/test split.

### 📊 Phase 2 Observations
Chapter distribution, skip chapter decision, training examples per
chapter, Stage-1 data advantage over E-002.

### 🧭 Phase 3a: Stage-1 Setup
Tokenisation, E-001 model loading, 22-way head replacement.

### 📊 Phase 3b: Stage-1 Trainer Configuration
TrainingArguments, TensorBoard and MLflow monitoring commands.

### 🚀 Phase 3c: Stage-1 Training Ignition
10-epoch chapter router training. Best checkpoint: epoch 9.

### 📊 Stage-1 Interpretation
Training curve analysis, E-001 initialisation impact, comparison
with E-002 flat chapter accuracy.

### 📊 Phase 3b Evaluation: Stage-1 Test Set
Definitive test routing accuracy, per-chapter routing breakdown,
routing error budget for Stage-2.

### 📊 Stage-1 Test Set Interpretation
Per-chapter routing reliability, problem chapters (T, Z, S),
Stage-2 priority order.

### 🔬 Phase 4a: Stage-2 Data Preparation
Per-chapter dataset filtering, label encoders, tokenisation,
skip chapter fallback predictions.

### 📊 Phase 4b: Stage-2 Trainer Configuration
`train_chapter_resolver()` function, fresh Bio_ClinicalBERT
rationale, TensorBoard monitoring.

### 🚀 Phase 4c: Stage-2 Training Loop
19 chapter resolvers trained in priority order. All 19 saved
successfully. Weighted val accuracy: 13.6%.

### 🎯 Phase 5: End-to-End Pipeline Evaluation
Full two-stage pipeline evaluation on test set. E2E accuracy:
10.6%, Macro F1: 0.070.

### 📊 E-003 Results: Interpretation
Stage-2 failure analysis, data fragmentation diagnosis, E-004a fix.

### 🏆 Phase 6: Registry Promotion
Stage-1 router and experiment metadata saved to registry.
MLflow run closed.

---

### 🎯 Experiment Objective

Implement a hierarchical two-stage ICD-10 prediction pipeline
motivated by E-002's finding that 82.9% of predictions land in the
correct chapter while only 46.9% reach the correct specific code.

**Official E-003 Results:**

| Stage | Metric | Value |
|---|---|---|
| Stage-1 | Chapter routing accuracy | 95.3% |
| Stage-1 | Chapter Macro F1 | 0.959 |
| Stage-2 | Within-chapter accuracy | 11.1% |
| End-to-end | Accuracy | 10.6% |
| End-to-end | Macro F1 | 0.070 |

**Key finding:** Stage-1 significantly outperformed E-002's flat
chapter accuracy (+12.4pp). Stage-2 underperformed due to training
data fragmentation and fresh Bio_ClinicalBERT initialisation.
Fix: initialise Stage-2 from E-002 registry model in E-004a.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 🧠 Hierarchical Two-Stage ICD-10 Classification (E-003)

This notebook implements the third modelling experiment on the MedSynth
Gold layer. Having established flat classification baselines in E-001
(ICD-3, Macro F1 = 0.760) and E-002 (ICD-10, Macro F1 = 0.352), we now
implement the hierarchical two-stage architecture directly motivated by
the E-002 chapter confusion analysis.

## 🎯 Motivation

The E-002 chapter-level confusion analysis produced a decisive finding:

- **Chapter-level accuracy: 82.9%** — the model correctly identifies
  the ICD-10 clinical chapter in 83% of predictions
- **Code-level accuracy: 46.9%** — but only resolves the specific code
  in 47% of predictions
- **67.8% of all errors stay within the correct chapter** — the model
  knows the clinical domain but struggles with fine-grained within-chapter
  discrimination

This gap between chapter-level and code-level accuracy is the structural
problem a hierarchical architecture is designed to solve.

## 🏗️ Architecture

| Stage | Task | Classes | Training data | Starting point |
|---|---|---|---|---|
| **Stage 1** | Chapter router | 22 ICD-10 chapters | 9,660 billable records | E-001 registry model |
| **Stage 2** | Within-chapter resolver | Varies per chapter | Chapter-filtered subset | Fresh Bio_ClinicalBERT |

**Stage 1** routes each clinical note to its ICD-10 chapter (A–Z).
Initialised from the E-001 registry model which was pre-trained on ICD-3
categories — already chapter-aware. With ~440 training examples per
chapter, this should substantially exceed E-002's flat 82.9% chapter
accuracy.

**Stage 2** resolves the specific ICD-10 code within the routed chapter.
Each chapter gets its own resolver trained on chapter-filtered records,
dramatically reducing the effective label space compared to E-002's
1,926-way flat classifier.

Priority chapters for Stage-2 (lowest E-002 flat accuracy):

| Chapter | Classes | E-002 flat accuracy | Training records |
|---|---|---|---|
| Z | 263 | 48.9% | ~685 |
| R | 196 | 63.2% | ~435 |
| O | 63 | 75.0% | ~180 |
| M | 222 | 94.2% | ~520 |

## 🔬 Hypothesis

Routing through a high-accuracy chapter classifier before resolving
within-chapter codes will outperform E-002's flat ICD-10 classification
on end-to-end accuracy and Macro F1.

## 🛠️ Technical Stack

- **Model:** `emilyalsentzer/Bio_ClinicalBERT`
- **Stage-1 initialisation:** E-001 registry model
  (`outputs/evaluations/registry/E-001_Baseline_ICD3/model/`)
- **Framework:** HuggingFace Transformers + Trainer API
- **Metrics:** End-to-end Macro F1, Accuracy, Top-5 Accuracy
- **Experiment tracking:** MLflow with SQLite backend
- **Data:** MedSynth Gold layer Parquet — 9,660 billable records

---

**Next:** Phase 1 — Experiment configuration and registry path discovery.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 1: Experiment Configuration (E-003)

Configuration-driven setup consistent with notebooks 02 and 03.
E-003 introduces two new cfg elements not present in earlier notebooks:
separate hyperparameter sets for Stage-1 and Stage-2, and registry
path discovery for the E-001 model used to initialise the chapter router.

### Key Controls

- **Data source:** Same Gold layer Parquet as E-001 and E-002.
- **Records:** 9,660 billable records only — same filter as E-002.
- **Stage-1 (chapter router):** 10 epochs, 22-way chapter classifier,
  initialised from the E-001 registry model. Fewer epochs needed — 22
  classes with ~440 examples each converges much faster than 1,926
  classes with ~4 examples each.
- **Stage-2 (within-chapter resolvers):** 20 epochs per chapter,
  consistent with E-002. Each resolver sees only records from its
  chapter, dramatically reducing the effective label space.
- **Priority chapter order:** Lowest E-002 flat accuracy first — Z
  (48.9%), R (63.2%), O (75.0%) — so the most impactful resolvers
  are trained first.
- **E-001 registry path verified at startup** — raises `FileNotFoundError`
  immediately if the Stage-1 initialisation model is missing.

</div>

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENT CONFIGURATION (E-003 HIERARCHICAL ICD-10)
# ==============================================================================
import sys
import torch
import mlflow
import polars as pl
from pathlib import Path

# ------------------------------------------------------------------------------
# 1. BOOTSTRAP: DISCOVER PROJECT ROOT (enables 'import src')
# ------------------------------------------------------------------------------
print("🔍 Discovering project root...")

try:
    current = Path.cwd()
    while current != current.parent:
        if (current / "artifacts.yaml").exists():
            PROJECT_ROOT = current.resolve()
            break
        current = current.parent
    else:
        raise FileNotFoundError(
            "Could not find artifacts.yaml in current or parent directories."
        )
    print(f"   📍 Project root: .../{PROJECT_ROOT.name}")
except FileNotFoundError as e:
    print(f"❌ CRITICAL: {e}")
    raise

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"   📦 Project root added to sys.path")

# ------------------------------------------------------------------------------
# 2. IMPORT CONFIG
# ------------------------------------------------------------------------------
from src.config import config

if Path(config.project_root) != PROJECT_ROOT:
    raise RuntimeError(
        f"Project root mismatch!\n"
        f"  Bootstrap found: {PROJECT_ROOT}\n"
        f"  Config reports:  {config.project_root}"
    )

print(f"   ✅ Config loaded from: {config.config_path}")

# ------------------------------------------------------------------------------
# 3. MLFLOW SQLITE BACKEND
# ------------------------------------------------------------------------------
DB_PATH = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# ------------------------------------------------------------------------------
# 4. EXPERIMENT PARAMETERS
# ------------------------------------------------------------------------------
cfg = {
    "experiment_id":       "E-003",
    "experiment_name":     "E-003_Hierarchical_ICD10",
    "description":         "Two-stage hierarchical ICD-10 | Stage-1 chapter router | Stage-2 within-chapter resolvers",
    "model_name":          "emilyalsentzer/Bio_ClinicalBERT",
    "payload_type":        "note_only",
    "label_scheme":        "hierarchical",
    "code_status_filter":  "billable",

    # Stage-1 (chapter router) parameters
    "stage1_num_epochs":   10,             # 22 classes, ~440 examples each — converges faster
    "stage1_learning_rate": 2e-5,
    "stage1_batch_size":   16,

    # Stage-2 (within-chapter resolvers) parameters
    "stage2_num_epochs":   20,             # consistent with E-002
    "stage2_learning_rate": 2e-5,
    "stage2_batch_size":   16,

    "max_length":          512,
    "weight_decay":        0.01,
    "warmup_ratio":        0.1,
    "use_special_tokens":  False,
    "seed":                42,

    # E-001 registry model used to initialise Stage-1
    "stage1_init_model":   "E-001_Baseline_ICD3",

    # Priority chapters for Stage-2 (lowest E-002 flat accuracy first)
    "stage2_priority_chapters": ["Z", "R", "O", "B", "A", "T", "G", "S",
                                  "D", "E", "L", "H", "K", "F", "J", "N",
                                  "I", "C", "M", "P", "Q", "U"],
}

# ------------------------------------------------------------------------------
# 5. LOCATE GOLD LAYER PARQUET
# ------------------------------------------------------------------------------
gold_dir      = config.resolve_path("data", "gold")
parquet_files = sorted(gold_dir.glob("medsynth_gold_apso_*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Gold layer Parquet found in {gold_dir}\n"
        f"Please run the EDA notebook (Phase 4) first."
    )

GOLD_PARQUET_PATH = parquet_files[-1]
print(f"\n   📁 Gold layer: {GOLD_PARQUET_PATH.name}")

# ------------------------------------------------------------------------------
# 6. LOCATE E-001 REGISTRY MODEL (Stage-1 initialisation)
# ------------------------------------------------------------------------------
REGISTRY_BASE = config.resolve_path("outputs", "evaluations") / "registry"
E001_MODEL_PATH = REGISTRY_BASE / cfg["stage1_init_model"] / "model"

if not E001_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"E-001 registry model not found at {E001_MODEL_PATH}\n"
        f"Please run notebook 02 through Phase 10 first."
    )

print(f"   📁 E-001 model:   {E001_MODEL_PATH.parent.name}/model/")

# ------------------------------------------------------------------------------
# 7. MLFLOW EXPERIMENT SETUP
# ------------------------------------------------------------------------------
mlflow.set_experiment(cfg["experiment_name"])

if mlflow.active_run():
    print(f"🔄 Closing active run: {mlflow.active_run().info.run_id}")
    mlflow.end_run()

run = mlflow.start_run(run_name=f"{cfg['experiment_id']}_Hierarchical")
mlflow.log_params({k: v for k, v in cfg.items()
                   if not isinstance(v, list)})
mlflow.log_param("gold_parquet",    GOLD_PARQUET_PATH.name)
mlflow.log_param("stage1_init",     cfg["stage1_init_model"])

# ------------------------------------------------------------------------------
# 8. DEVICE SETUP
# ------------------------------------------------------------------------------
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
mlflow.log_param("device", str(device))

# ------------------------------------------------------------------------------
# 9. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 1: Experiment Configuration",
    action="e003_config_initialised",
    details={
        "experiment_id":        cfg["experiment_id"],
        "model_name":           cfg["model_name"],
        "label_scheme":         cfg["label_scheme"],
        "code_status_filter":   cfg["code_status_filter"],
        "stage1_init_model":    cfg["stage1_init_model"],
        "stage1_num_epochs":    cfg["stage1_num_epochs"],
        "stage2_num_epochs":    cfg["stage2_num_epochs"],
        "gold_parquet":         GOLD_PARQUET_PATH.name,
        "device":               str(device),
        "mlflow_db":            str(DB_PATH),
    },
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"\n🔒 Experiment locked:  {cfg['experiment_name']}")
print(f"🚀 Acceleration:       {device.type.upper()}")
print(f"📊 MLflow backend:     {DB_PATH.name}")
print(f"✅ Configuration ready for ingestion.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## ⚙️ Phase 1b: Environment Setup & Imports (E-003)

Identical to notebooks 02 and 03 with one addition — separate output
directories for Stage-1 and Stage-2 are created at startup so each
stage's checkpoints, dashboards, and metrics are kept cleanly separated.

- `outputs/evaluations/E-003_Hierarchical_ICD10/stage1/` — chapter router
- `outputs/evaluations/E-003_Hierarchical_ICD10/stage2/` — within-chapter resolvers
- `outputs/evaluations/registry/E-003_Hierarchical_ICD10/` — final promoted artifacts

</div>

In [ ]:
# ==============================================================================
# PHASE 1b: ENVIRONMENT SETUP & IMPORTS (E-003)
# ==============================================================================

import os
import numpy as np
from pathlib import Path
from transformers import set_seed

# Integrity checks
assert "cfg" in globals(), \
    "❌ cfg not found. Run Phase 1 first."
assert "PROJECT_ROOT" in globals(), \
    "❌ PROJECT_ROOT not found. Run Phase 1 first."
assert "GOLD_PARQUET_PATH" in globals(), \
    "❌ GOLD_PARQUET_PATH not found. Run Phase 1 first."
assert "E001_MODEL_PATH" in globals(), \
    "❌ E001_MODEL_PATH not found. Run Phase 1 first."

# HuggingFace local cache
HF_CACHE_DIR = PROJECT_ROOT / "data" / "cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"]                = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"]           = str(HF_CACHE_DIR)
os.environ["HF_HUB_READ_TIMEOUT"]    = "120"
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Imports
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)

# Seed
active_seed = cfg.get("seed", 42)
set_seed(active_seed)

# Experiment output directories
EXP_DIR         = config.resolve_path("outputs", "evaluations") / cfg["experiment_name"]
STAGE1_DIR      = EXP_DIR / "stage1"
STAGE2_DIR      = EXP_DIR / "stage2"
REGISTRY_DIR    = config.resolve_path("outputs", "evaluations") / "registry" / cfg["experiment_name"]

for d in [EXP_DIR, STAGE1_DIR, STAGE2_DIR, REGISTRY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["TENSORBOARD_LOGGING_DIR"] = str(EXP_DIR / "tensorboard")

print(f"✅ Phase 1b complete")
print(f"   HF cache:        {HF_CACHE_DIR}")
print(f"   Experiment dir:  {EXP_DIR}")
print(f"   Stage-1 dir:     {STAGE1_DIR}")
print(f"   Stage-2 dir:     {STAGE2_DIR}")
print(f"   Seed:            {active_seed}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📥 Phase 2: Data Loading & Label Derivation (E-003)

Loads the Gold layer Parquet and derives both label columns needed for
the two-stage architecture from the same source data.

### Two Label Columns from One Source

| Column | Use | Classes | Derived from |
|---|---|---|---|
| `chapter_label` / `chapter_id` | Stage-1 router | 22 | `standard_icd10[0]` |
| `standard_icd10` / `icd10_id` | Stage-2 resolvers | 1,926 | `standard_icd10` |

### Split Strategy

The train/val/test split is **shared across both stages** — the same
7,728 / 966 / 966 records are used for Stage-1 and Stage-2 training
and evaluation. This ensures the end-to-end pipeline evaluation on the
test set is a fair comparison with E-002.

The first split is **stratified by chapter** rather than by ICD-10 code.
With ~440 records per chapter, stratification is reliable at the chapter
level — unlike E-002 where only 5 records per ICD-10 code made full
stratification impossible.

</div>

In [ ]:
# ==============================================================================
# PHASE 2: DATA LOADING & LABEL DERIVATION (E-003)
# ==============================================================================
# Purpose: Load the Gold layer Parquet, filter to billable records, and derive
# both label columns needed for the two-stage architecture:
#
#   chapter_label   — first character of standard_icd10 (22 classes)
#                     used for Stage-1 chapter router training
#   standard_icd10  — full ICD-10 code (1,926 classes)
#                     used for Stage-2 within-chapter resolver training
#
# Both stages share the same train/val/test split so the end-to-end
# pipeline evaluation on the test set is a fair comparison with E-002.
# ==============================================================================

import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------------------
# 1. LOAD GOLD LAYER
# ------------------------------------------------------------------------------
print(f"📂 Loading Gold layer: {GOLD_PARQUET_PATH.name}")

df_gold = pl.read_parquet(GOLD_PARQUET_PATH)

print(f"   ✅ Loaded: {len(df_gold):,} records, {len(df_gold.columns)} columns")

required_cols = {'id', 'apso_note', 'standard_icd10', 'code_status'}
missing_cols  = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(f"❌ Gold layer missing required columns: {missing_cols}")

print(f"   ✅ All required columns present")

# ------------------------------------------------------------------------------
# 2. FILTER TO BILLABLE RECORDS
# ------------------------------------------------------------------------------
print(f"\n📊 Full dataset code_status breakdown:")
for row in (df_gold.group_by("code_status")
            .agg(pl.len().alias("count"))
            .sort("count", descending=True)
            .iter_rows(named=True)):
    pct = row['count'] / len(df_gold) * 100
    print(f"   {row['code_status']:20s}: {row['count']:,} ({pct:.1f}%)")

df_gold = df_gold.filter(pl.col("code_status") == cfg["code_status_filter"])
print(f"\n   ✅ Filtered to '{cfg['code_status_filter']}': {len(df_gold):,} records")

# ------------------------------------------------------------------------------
# 3. DERIVE CHAPTER LABEL (Stage-1)
# ------------------------------------------------------------------------------
print(f"\n🔧 Deriving chapter labels (Stage-1)...")

df_gold = df_gold.with_columns(
    pl.col("standard_icd10").str.slice(0, 1).alias("chapter_label")
)

chapters      = sorted(df_gold["chapter_label"].unique().to_list())
chapter2id    = {ch: i for i, ch in enumerate(chapters)}
id2chapter    = {i: ch for ch, i in chapter2id.items()}
num_chapters  = len(chapters)

df_gold = df_gold.with_columns(
    pl.col("chapter_label")
    .replace(list(chapter2id.keys()), [chapter2id[k] for k in chapter2id.keys()])
    .cast(pl.Int64)
    .alias("chapter_id")
)

print(f"   ✅ Chapter labels derived: {num_chapters} ICD-10 chapters")
print(f"   📊 Chapters: {chapters}")

# Chapter distribution
print(f"\n   📊 Records per chapter:")
chapter_dist = (df_gold.group_by("chapter_label")
                .agg(pl.len().alias("count"))
                .sort("count", descending=True))
for row in chapter_dist.iter_rows(named=True):
    print(f"      {row['chapter_label']:4s}: {row['count']:,}")

# ------------------------------------------------------------------------------
# 4. ENCODE FULL ICD-10 LABELS (Stage-2)
# ------------------------------------------------------------------------------
print(f"\n🔧 Encoding ICD-10 labels (Stage-2)...")

all_icd10_labels = sorted(df_gold["standard_icd10"].unique().to_list())
icd10_label2id   = {label: idx for idx, label in enumerate(all_icd10_labels)}
icd10_id2label   = {idx: label for label, idx in icd10_label2id.items()}
num_icd10_labels = len(icd10_label2id)

df_gold = df_gold.with_columns(
    pl.col("standard_icd10")
    .replace(list(icd10_label2id.keys()),
             [icd10_label2id[k] for k in icd10_label2id.keys()])
    .cast(pl.Int64)
    .alias("icd10_id")
)

print(f"   ✅ ICD-10 labels encoded: {num_icd10_labels:,} classes")

# ------------------------------------------------------------------------------
# 5. TRAIN / VAL / TEST SPLIT (shared across both stages)
# ------------------------------------------------------------------------------
# Stratify by chapter_id for the first split — chapter has sufficient
# records per class (~440) for reliable stratification.
# Val/test split is random (same rationale as E-002).
# ------------------------------------------------------------------------------
print(f"\n✂️  Splitting dataset (80/10/10) — stratified by chapter...")

df_pd = df_gold.select([
    "id", "apso_note",
    "standard_icd10", "icd10_id",
    "chapter_label", "chapter_id",
    "code_status"
]).to_pandas()

train_df, temp_df = train_test_split(
    df_pd,
    test_size=0.2,
    stratify=df_pd["chapter_id"],   # stratify by chapter — 22 classes, ~440 each
    random_state=cfg["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=None,                   # not stratified — same rationale as E-002
    random_state=cfg["seed"]
)

print(f"   ✅ Split complete:")
print(f"      Train: {len(train_df):,} ({len(train_df)/len(df_pd)*100:.1f}%) — stratified by chapter")
print(f"      Val:   {len(val_df):,}  ({len(val_df)/len(df_pd)*100:.1f}%) — random")
print(f"      Test:  {len(test_df):,}  ({len(test_df)/len(df_pd)*100:.1f}%) — random")

# Verify chapter coverage in each split
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    unique_chapters = df["chapter_label"].nunique()
    print(f"      {name:5s} chapter coverage: {unique_chapters}/{num_chapters}")

# Log to MLflow
mlflow.log_params({
    "num_chapters":        num_chapters,
    "num_icd10_labels":    num_icd10_labels,
    "train_size":          len(train_df),
    "val_size":            len(val_df),
    "test_size":           len(test_df),
    "train_stratified":    True,
    "val_test_stratified": False,
})

# ------------------------------------------------------------------------------
# 6. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 2: Data Loading & Label Derivation",
    action="data_loaded_and_split",
    details={
        "gold_parquet":        GOLD_PARQUET_PATH.name,
        "total_records":       len(df_gold),
        "code_status_filter":  cfg["code_status_filter"],
        "num_chapters":        num_chapters,
        "num_icd10_labels":    num_icd10_labels,
        "train_size":          len(train_df),
        "val_size":            len(val_df),
        "test_size":           len(test_df),
        "train_stratified_by": "chapter_id",
    },
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 2 complete: {num_chapters}-way Stage-1 + {num_icd10_labels:,}-way Stage-2 ready")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

### 📊 Phase 2 Observations

**22 ICD-10 chapters across 9,660 billable records.** Chapter sizes
range dramatically from Z (1,320 records, 263 classes) to U (5 records,
1 class), reflecting the MedSynth IQVIA disease distribution.

**Within-chapter record density is uniform at ~5 records per ICD-10
code** — MedSynth's uniform sampling design applies within chapters
as well as globally. Stage-2 resolvers face the same ~4 training
examples per class challenge as E-002's flat classifier.

**Three chapters are too small for meaningful Stage-2 training:**

| Chapter | Records | Classes | Action |
|---|---|---|---|
| U | 5 | 1 | Skip — single class, always predict it |
| Q | 20 | 4 | Skip or use majority class predictor |
| P | 25 | 5 | Skip or use majority class predictor |

**Val coverage: 20/22 chapters** — P and Q are absent from the
validation set (too few records for random 10% split to guarantee
representation). Test coverage is 22/22.

**The Stage-1 chapter router has a significant data advantage over
E-002's flat classifier** — ~440 records per chapter vs ~5 records
per ICD-10 code. This is why Stage-1 should substantially outperform
E-002's 82.9% flat chapter accuracy.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🧭 Phase 3a: Stage-1 Setup — Tokenisation & Model Initialisation

Loads the E-001 registry model and tokenizer, builds the Stage-1
DatasetDict with 22-way chapter labels, tokenises all splits, and
replaces the classification head for 22-way chapter classification.

The E-001 model (pre-trained on 675 ICD-3 categories) is already
implicitly chapter-aware — every ICD-3 stem begins with the chapter
letter. Fine-tuning for 22-way chapter classification sharpens an
existing capability rather than learning from scratch.

The E-001 675-way head is replaced with a fresh 22-way head using
`ignore_mismatched_sizes=True` — the same head-swap pattern used
in E-001 and E-002. BERT encoder weights are preserved.

</div>

In [ ]:
# ==============================================================================
# PHASE 3a: STAGE-1 SETUP — TOKENISATION & MODEL INITIALISATION
# ==============================================================================

import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from src.evaluation import hf_compute_metrics

# ------------------------------------------------------------------------------
# 1. LOAD TOKENIZER FROM E-001 REGISTRY
# ------------------------------------------------------------------------------
print(f"📥 Loading tokenizer from E-001 registry...")

tokenizer = AutoTokenizer.from_pretrained(
    str(E001_MODEL_PATH),
    cache_dir=str(HF_CACHE_DIR)
)

print(f"   ✅ Tokenizer loaded: vocab size {len(tokenizer):,}")

# ------------------------------------------------------------------------------
# 2. BUILD STAGE-1 DATASETDICT (chapter-level labels)
# ------------------------------------------------------------------------------
print(f"\n🔧 Building Stage-1 DatasetDict (22-way chapter classification)...")

class_names_chapter = sorted(list(chapter2id.keys()))

features_stage1 = Features({
    'text':  Value('string'),
    'label': ClassLabel(names=class_names_chapter)
})

stage1_datasets = DatasetDict({
    "train": Dataset.from_dict(
        {"text": train_df["apso_note"].tolist(),
         "label": train_df["chapter_id"].tolist()},
        features=features_stage1
    ),
    "val": Dataset.from_dict(
        {"text": val_df["apso_note"].tolist(),
         "label": val_df["chapter_id"].tolist()},
        features=features_stage1
    ),
    "test": Dataset.from_dict(
        {"text": test_df["apso_note"].tolist(),
         "label": test_df["chapter_id"].tolist()},
        features=features_stage1
    ),
})

print(f"   ✅ Stage-1 DatasetDict:")
for split in ["train", "val", "test"]:
    n        = len(stage1_datasets[split])
    n_classes = stage1_datasets[split].features['label'].num_classes
    print(f"      {split:5s}: {n:,} records, {n_classes} classes")

# ------------------------------------------------------------------------------
# 3. TOKENISE STAGE-1 DATASETS
# ------------------------------------------------------------------------------
print(f"\n🔄 Tokenising Stage-1 dataset (max_length={cfg['max_length']})...")

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg["max_length"]
    )

stage1_tokenized = stage1_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"]
)

stage1_tokenized.set_format("torch")
print(f"   ✅ Tokenisation complete")
print(f"   📊 Columns: {stage1_tokenized['train'].column_names}")

sample_ids    = stage1_tokenized["train"][0]["input_ids"]
first_content = tokenizer.decode(sample_ids[1:15], skip_special_tokens=True)
print(f"   🔬 Content tokens 1–14: {first_content}")

# ------------------------------------------------------------------------------
# 4. LOAD E-001 MODEL — REPLACE HEAD FOR 22-WAY CHAPTER CLASSIFICATION
# ------------------------------------------------------------------------------
print(f"\n🧠 Loading E-001 model, replacing head for 22-way chapter classification...")

stage1_model = AutoModelForSequenceClassification.from_pretrained(
    str(E001_MODEL_PATH),
    num_labels=num_chapters,
    id2label=id2chapter,
    label2id=chapter2id,
    ignore_mismatched_sizes=True,
    cache_dir=str(HF_CACHE_DIR)
)

stage1_model.to(device)

print(f"   ✅ Stage-1 model: {stage1_model.num_labels}-way chapter classifier")
print(f"   ✅ Device: {next(stage1_model.parameters()).device}")
print(f"   ✅ Vocab:  {stage1_model.get_input_embeddings().weight.shape[0]:,}")

assert stage1_model.num_labels == num_chapters, \
    f"❌ Head mismatch: {stage1_model.num_labels} vs {num_chapters}"

print(f"\n✅ Phase 3a complete — run Phase 3b to configure trainer")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 3b: Stage-1 Trainer Configuration

Configures the Trainer for the 22-way chapter router. Identical pattern
to notebooks 02 and 03 — Macro F1 as primary metric, `save_total_limit=3`,
`dataloader_pin_memory=False` for MPS stability.

**Epochs:** 10 — with ~440 training examples per chapter and only 22
classes, convergence is expected well before epoch 10.

**Baseline to beat:** E-002 flat chapter accuracy of 82.9%.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b: STAGE-1 TRAINER CONFIGURATION
# ==============================================================================

from transformers import TrainingArguments, DataCollatorWithPadding, Trainer

# Warmup steps
stage1_total_steps  = (
    len(stage1_tokenized["train"]) // cfg["stage1_batch_size"]
) * cfg["stage1_num_epochs"]
stage1_warmup_steps = int(cfg["warmup_ratio"] * stage1_total_steps)

print(f"   📊 Stage-1 total steps:  {stage1_total_steps:,}")
print(f"   📊 Stage-1 warmup steps: {stage1_warmup_steps:,}")

STAGE1_CHECKPOINT_DIR = STAGE1_DIR / "checkpoints"
STAGE1_TB_DIR         = STAGE1_DIR / "tensorboard"
STAGE1_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_TB_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TENSORBOARD_LOGGING_DIR"] = str(STAGE1_TB_DIR)

stage1_args = TrainingArguments(
    output_dir              = str(STAGE1_CHECKPOINT_DIR),
    eval_strategy           = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "macro_f1",
    greater_is_better       = True,
    save_total_limit        = 3,
    num_train_epochs            = cfg["stage1_num_epochs"],
    per_device_train_batch_size = cfg["stage1_batch_size"],
    learning_rate               = cfg["stage1_learning_rate"],
    weight_decay                = cfg["weight_decay"],
    warmup_steps                = stage1_warmup_steps,
    logging_steps               = 20,
    report_to                   = ["tensorboard", "mlflow"],
    seed                        = cfg["seed"],
    fp16                        = False,
    dataloader_pin_memory       = False,
)

stage1_trainer = Trainer(
    model            = stage1_model,
    args             = stage1_args,
    train_dataset    = stage1_tokenized["train"],
    eval_dataset     = stage1_tokenized["val"],
    processing_class = tokenizer,
    data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics  = hf_compute_metrics,
)

print(f"\n✅ Stage-1 trainer ready")
print(f"   Train:   {len(stage1_tokenized['train']):,} records")
print(f"   Val:     {len(stage1_tokenized['val']):,} records")
print(f"   Epochs:  {cfg['stage1_num_epochs']}")
print(f"   Classes: {num_chapters} ICD-10 chapters")
print(f"   Init:    E-001 registry model")

active_run = mlflow.active_run()
if active_run:
    print(f"\n✅ Active MLflow run: {active_run.info.run_id}")
    mlflow.log_params({
        "stage1_checkpoint_dir":  str(STAGE1_CHECKPOINT_DIR),
        "stage1_tensorboard_dir": str(STAGE1_TB_DIR),
        "stage1_total_steps":     stage1_total_steps,
        "stage1_warmup_steps":    stage1_warmup_steps,
    })

print(f"\n{'='*70}")
print(f"📈 TENSORBOARD (Stage-1):")
print(f"   tensorboard --logdir '{STAGE1_TB_DIR}' --port 6006")
print(f"\n📊 MLFLOW UI:")
print(f"   mlflow ui --backend-store-uri sqlite:///{PROJECT_ROOT}/mlflow.db --port 5001")
print(f"{'='*70}")
print(f"\n✅ Phase 3b complete — open TensorBoard then run Phase 3c to start training")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🚀 Phase 3c: Stage-1 Training Ignition

Executes the 10-epoch Stage-1 chapter router training run.
Best checkpoint selected by Macro F1, auto-reloaded at completion.

Expected duration: approximately 40–60 minutes on Apple Silicon MPS.

</div>

In [ ]:
# # ==============================================================================
# # PHASE 3c: STAGE-1 TRAINING IGNITION
# # ==============================================================================
# import json

# print(f"🚀 Igniting Stage-1 Chapter Router training...")
# print(f"   22-way chapter classification")
# print(f"   Init: E-001 registry model")
# print(f"   Target: > 82.9% chapter accuracy (E-002 flat baseline)")

# stage1_result = stage1_trainer.train()

# print(f"\n✅ Stage-1 training complete")
# print(f"   Training loss: {stage1_result.training_loss:.4f}")
# print(f"   Runtime:       {stage1_result.metrics.get('train_runtime', 0):.1f}s")
# print(f"   Samples/sec:   {stage1_result.metrics.get('train_samples_per_second', 0):.1f}")

# # Best epoch metrics from log history
# stage1_val_metrics = {}
# for log in stage1_trainer.state.log_history:
#     if 'eval_macro_f1' in log:
#         stage1_val_metrics = log

# print(f"\n📊 Stage-1 Best Val Results:")
# for k, v in stage1_val_metrics.items():
#     if k.startswith("eval_"):
#         print(f"   {k}: {v:.4f}")

# # Save model and chapter mapping
# STAGE1_MODEL_DIR = STAGE1_DIR / "model"
# stage1_trainer.save_model(str(STAGE1_MODEL_DIR))
# tokenizer.save_pretrained(str(STAGE1_MODEL_DIR))

# with open(STAGE1_DIR / "chapter_mapping.json", "w") as f:
#     json.dump({
#         "chapter2id":  chapter2id,
#         "id2chapter":  {str(k): v for k, v in id2chapter.items()},
#         "num_chapters": num_chapters
#     }, f, indent=4)

# print(f"\n   ✅ Stage-1 model saved: {STAGE1_MODEL_DIR}")

# mlflow.log_metrics({
#     "stage1_train_loss":   stage1_result.training_loss,
#     "stage1_val_macro_f1": stage1_val_metrics.get("eval_macro_f1", 0),
#     "stage1_val_accuracy": stage1_val_metrics.get("eval_accuracy", 0),
#     "stage1_val_top5":     stage1_val_metrics.get("eval_top_5_accuracy", 0),
# })

# config.log_event(
#     phase="Phase 3c: Stage-1 Training",
#     action="stage1_training_complete",
#     details={
#         "init_model":    cfg["stage1_init_model"],
#         "num_chapters":  num_chapters,
#         "num_epochs":    cfg["stage1_num_epochs"],
#         "train_loss":    stage1_result.training_loss,
#         "val_macro_f1":  stage1_val_metrics.get("eval_macro_f1", 0),
#         "val_accuracy":  stage1_val_metrics.get("eval_accuracy", 0),
#         "model_path":    str(STAGE1_MODEL_DIR),
#     },
#     notebook="04-Model_Hierarchical_ICD10"
# )

# print(f"\n📝 Audit trail updated")
# print(f"✅ Phase 3c complete: Stage-1 chapter router trained and saved")

In [ ]:
# ==============================================================================
# PHASE 3c: STAGE-1 CHAPTER ROUTER — TRAIN OR LOAD
# ==============================================================================
import json

STAGE1_MODEL_DIR = STAGE1_DIR / "model"
FORCE_RETRAIN    = False  # set to True to force retraining even if model exists

if STAGE1_MODEL_DIR.exists() and not FORCE_RETRAIN:
    # ------------------------------------------------------------------
    # LOAD existing model — skip retraining
    # ------------------------------------------------------------------
    print(f"📥 Stage-1 model found on disk — loading (set FORCE_RETRAIN=True to retrain)")

    from transformers import AutoModelForSequenceClassification

    stage1_model = AutoModelForSequenceClassification.from_pretrained(
        str(STAGE1_MODEL_DIR)
    )
    stage1_model.to(device)
    stage1_model.eval()

    # Rebuild stage1_trainer for predict() calls in Phase 3b eval and Phase 5
    from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

    _dummy_args = TrainingArguments(
        output_dir            = str(STAGE1_DIR / "checkpoints"),
        fp16                  = False,
        dataloader_pin_memory = False,
        log_level             = "error",
        disable_tqdm          = True,
        report_to             = "none",
    )
    stage1_trainer = Trainer(
        model            = stage1_model,
        args             = _dummy_args,
        processing_class = tokenizer,
        data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
    )

    with open(STAGE1_DIR / "chapter_mapping.json") as f:
        ch_map = json.load(f)

    print(f"   ✅ Stage-1 model loaded: {stage1_model.num_labels}-way classifier")
    print(f"   ✅ Device: {next(stage1_model.parameters()).device}")
    print(f"   ✅ Chapter mapping loaded: {ch_map['num_chapters']} chapters")
    print(f"\n✅ Phase 3c complete — Stage-1 ready (loaded from disk, not retrained)")

else:
    # ------------------------------------------------------------------
    # TRAIN from scratch
    # ------------------------------------------------------------------
    print(f"🚀 Igniting Stage-1 Chapter Router training...")
    print(f"   22-way chapter classification")
    print(f"   Init: E-001 registry model")
    print(f"   Target: > 82.9% chapter accuracy (E-002 flat baseline)")

    stage1_result = stage1_trainer.train()

    print(f"\n✅ Stage-1 training complete")
    print(f"   Training loss: {stage1_result.training_loss:.4f}")
    print(f"   Runtime:       {stage1_result.metrics.get('train_runtime', 0):.1f}s")
    print(f"   Samples/sec:   {stage1_result.metrics.get('train_samples_per_second', 0):.1f}")

    stage1_val_metrics = {}
    for log in stage1_trainer.state.log_history:
        if 'eval_macro_f1' in log:
            stage1_val_metrics = log

    print(f"\n📊 Stage-1 Best Val Results:")
    for k, v in stage1_val_metrics.items():
        if k.startswith("eval_"):
            print(f"   {k}: {v:.4f}")

    STAGE1_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    stage1_trainer.save_model(str(STAGE1_MODEL_DIR))
    tokenizer.save_pretrained(str(STAGE1_MODEL_DIR))

    with open(STAGE1_DIR / "chapter_mapping.json", "w") as f:
        json.dump({
            "chapter2id":   chapter2id,
            "id2chapter":   {str(k): v for k, v in id2chapter.items()},
            "num_chapters": num_chapters
        }, f, indent=4)

    print(f"\n   ✅ Stage-1 model saved: {STAGE1_MODEL_DIR}")

    mlflow.log_metrics({
        "stage1_train_loss":   stage1_result.training_loss,
        "stage1_val_macro_f1": stage1_val_metrics.get("eval_macro_f1", 0),
        "stage1_val_accuracy": stage1_val_metrics.get("eval_accuracy", 0),
        "stage1_val_top5":     stage1_val_metrics.get("eval_top_5_accuracy", 0),
    })

    config.log_event(
        phase="Phase 3c: Stage-1 Training",
        action="stage1_training_complete",
        details={
            "init_model":    cfg["stage1_init_model"],
            "num_chapters":  num_chapters,
            "num_epochs":    cfg["stage1_num_epochs"],
            "train_loss":    stage1_result.training_loss,
            "val_macro_f1":  stage1_val_metrics.get("eval_macro_f1", 0),
            "val_accuracy":  stage1_val_metrics.get("eval_accuracy", 0),
            "model_path":    str(STAGE1_MODEL_DIR),
        },
        notebook="04-Model_Hierarchical_ICD10"
    )

    print(f"\n📝 Audit trail updated")
    print(f"✅ Phase 3c complete: Stage-1 chapter router trained and saved")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Stage-1 Chapter Router: Results & Interpretation

### Final Performance (10 Epochs, 22-way Chapter Classification)

| Metric | Value | Context |
|---|---|---|
| Macro F1 | 0.968 | Equal weight to all 22 ICD-10 chapters |
| Accuracy | 95.5% | Chapter-level routing accuracy |
| Top-5 Accuracy | 98.6% | Correct chapter in top 5 predictions |
| Final val loss | 0.296 | Mild overfitting — val metrics still positive |
| Best epoch | 9 | Auto-reloaded at completion |
| Runtime | 73 min | 17.6 samples/sec on Apple Silicon MPS |

---

### What the Numbers Mean

**Macro F1 of 0.968** on a 22-way classification task with ~440 training
examples per chapter is exceptional. Every ICD-10 chapter is being
correctly identified with near-perfect reliability. The chapter router
has learned the clinical language signatures of all 22 diagnostic domains.

**Accuracy of 95.5%** versus E-002's flat chapter accuracy of 82.9% —
a +12.6pp improvement from training specifically for chapter routing
rather than using flat ICD-10 predictions as a proxy. This confirms the
core hypothesis: a dedicated chapter router substantially outperforms
implicit chapter-level accuracy from a flat classifier.

**Top-5 Accuracy of 98.6%** is the most critical metric for the pipeline.
In a hierarchical system, a Stage-1 error propagates directly to Stage-2
— if the wrong chapter is selected, Stage-2 cannot recover the correct
ICD-10 code regardless of its own accuracy. With 98.6% of records having
the correct chapter in the top 5, the pipeline has exceptional routing
reliability.

---

### TensorBoard Curve Analysis

**eval/accuracy** — rapid rise from 87.5% at epoch 1 to 94.9% at epoch
2, then steady improvement to 95.5%. The E-001 initialisation gave the
router a massive head start — epoch 1 already substantially exceeds
E-002's flat 82.9% baseline. ✅

**eval/loss** — minimum at epoch 3 (0.224), then gradually rising to
0.296 at epoch 10. Val loss diverging from train loss from epoch 3
onwards indicates mild overfitting. However val accuracy and Macro F1
continued improving through epoch 9, so the overfitting did not harm
the productive training window. ✅

**eval/macro_f1** — steep rise from 0.792 at epoch 1 to 0.912 at epoch
2, then continued improvement to 0.968 at epoch 9. The near-linear
improvement across all 10 epochs suggests the model had not fully
converged — additional epochs might push Macro F1 above 0.97. ✅

**eval/top_5_accuracy** — reaches 0.988 by epoch 2 and stays there.
The chapter router finds the correct chapter in its top 5 predictions
for 98.8% of records almost immediately, confirming the E-001 encoder
already has strong chapter-level representations. ✅

**train/loss** — steep descent from ~3.0 to near 0 by epoch 8, then
essentially flat. The model memorised the training patterns for the
22-chapter task. The train/val loss gap confirms mild but not harmful
overfitting. ✅

**train/learning_rate** — correct warmup/decay schedule. Rises over
the first 483 steps then decays linearly to 0 at step 4,830. ✅

---

### Why E-001 Initialisation Worked So Well

The dramatic jump from E-002's 82.9% flat chapter accuracy to Stage-1's
95.5% dedicated chapter accuracy is directly attributable to the E-001
initialisation. Three factors explain this:

1. **E-001 already learned ICD-3 groupings.** ICD-3 categories (e.g.
   `M25`, `N39`) are named by their first character (the chapter letter)
   followed by two digits. The E-001 encoder learned to associate clinical
   note language with 675 ICD-3 categories, implicitly learning chapter
   boundaries in the process.

2. **Chapter routing is a simpler task than ICD-3 classification.** The
   Stage-1 task (22 classes, ~440 examples each) has 30x more training
   examples per class than E-001 (675 classes, ~12 examples each). The
   model is being asked to do less with more data.

3. **The classification head swap is efficient.** Replacing the 675-way
   head with a fresh 22-way head means the BERT encoder weights are
   preserved — only the output mapping needs to be learned, not the
   clinical language representations.

---

### Stage-1 vs E-002 Flat Chapter Performance

| Metric | E-002 flat | Stage-1 dedicated | Improvement |
|---|---|---|---|
| Chapter accuracy | 82.9% | 95.5% | +12.6pp |
| Chapter Macro F1 | ~0.74 (adj.) | 0.968 | +0.228 |
| Training examples/class | ~440 | ~440 | Same |
| Epochs | 20 | 10 | Half |

The improvement is entirely structural — same data, same model family,
same hardware. The dedicated chapter router simply has the right objective
function and the right initialisation.

---

### Implications for Stage-2

With Stage-1 achieving 95.5% chapter routing accuracy, the end-to-end
pipeline error budget is well-bounded:

- **4.5% of records** will be routed to the wrong chapter and cannot
  be recovered by Stage-2 regardless of its performance
- **95.5% of records** will reach the correct Stage-2 resolver

If Stage-2 achieves 55% within-chapter accuracy (comparable to E-002's
within-chapter performance), the end-to-end pipeline accuracy would be
approximately 0.955 × 0.55 ≈ **52.5%** — already exceeding E-002's
flat 46.9%.

> **Stage-1 Status: Complete.** Chapter router saved to
> `stage1/model/`. Macro F1 = 0.968, Accuracy = 95.5%.
> Stage-2 within-chapter resolver training is next.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 3b Evaluation: Stage-1 Test Set Performance

Evaluates the best Stage-1 chapter router checkpoint on the held-out
test set. This establishes the definitive routing accuracy that bounds
the end-to-end pipeline performance — any record misrouted by Stage-1
cannot be recovered by Stage-2.

The per-chapter breakdown identifies which chapters the router handles
reliably vs which chapters will contribute disproportionately to
end-to-end errors, informing the priority order for Stage-2 training.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b EVALUATION: STAGE-1 TEST SET PERFORMANCE
# ==============================================================================
# Evaluates the best Stage-1 chapter router on the held-out test set.
# Produces the definitive Stage-1 routing accuracy before Stage-2 begins.
# ==============================================================================

import numpy as np
from collections import Counter

print(f"🔍 Evaluating Stage-1 router on test set ({len(stage1_tokenized['test']):,} records)...")

stage1_test_output = stage1_trainer.predict(stage1_tokenized["test"])
stage1_y_true      = stage1_test_output.label_ids
stage1_y_pred      = np.argmax(stage1_test_output.predictions, axis=-1)

print(f"\n📊 Stage-1 Test Results:")
for k, v in stage1_test_output.metrics.items():
    print(f"   {k}: {v:.4f}")

# Chapter-level breakdown
print(f"\n📊 Per-chapter routing accuracy (test set):")
print(f"   {'Chapter':8s}  {'True':>6s}  {'Correct':>8s}  {'Accuracy':>9s}")
print(f"   {'─'*36}")

chapter_test_stats = {}
for ch_id, ch_name in sorted(id2chapter.items()):
    mask      = stage1_y_true == ch_id
    n_true    = mask.sum()
    n_correct = (stage1_y_pred[mask] == ch_id).sum()
    accuracy  = n_correct / n_true if n_true > 0 else 0
    chapter_test_stats[ch_name] = {
        "n_true": int(n_true),
        "n_correct": int(n_correct),
        "accuracy": float(accuracy)
    }
    print(f"   {ch_name:8s}  {n_true:>6,}  {n_correct:>8,}  {accuracy:>8.1%}")

# Overall routing error budget


stage1_test_accuracy = (stage1_y_pred == stage1_y_true).mean()
stage1_test_f1       = stage1_test_output.metrics.get("test_macro_f1",
                       (stage1_y_pred == stage1_y_true).mean())



print(f"\n📊 Routing error budget for Stage-2:")
print(f"   Correctly routed:   {stage1_test_accuracy:.1%} of test records reach correct Stage-2 resolver")
print(f"   Misrouted:          {1-stage1_test_accuracy:.1%} of test records cannot be recovered by Stage-2")

# Log to MLflow
mlflow.log_metrics({
    "stage1_test_accuracy": stage1_test_accuracy,
    "stage1_test_macro_f1": stage1_test_f1,
    "stage1_test_top5":     stage1_test_output.metrics.get("test_top_5_accuracy", 0),
})

# Save chapter test stats
import json
with open(STAGE1_DIR / "chapter_test_stats.json", "w") as f:
    json.dump(chapter_test_stats, f, indent=4)

config.log_event(
    phase="Phase 3b Eval: Stage-1 Test",
    action="stage1_test_evaluation_complete",
    details={
        "test_accuracy": stage1_test_accuracy,
        "test_macro_f1": stage1_test_f1,
        "chapter_stats": chapter_test_stats,
    },
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 3b evaluation complete — ready for Stage-2")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Stage-1 Test Set Interpretation

### Official Stage-1 Results

| Metric | Val | Test |
|---|---|---|
| Macro F1 | 0.968 | 0.959 |
| Accuracy | 95.5% | 95.3% |
| Top-5 Accuracy | 98.6% | 98.3% |
| Val/Test gap | — | Minimal — no overfitting |

Val and test results are consistent — the chapter router generalises
cleanly to unseen data.

---

### Per-Chapter Routing Reliability

**Perfect or near-perfect routing (≥97%):**
A, C, E, F, H, I, M, N, O, P, Q, U — these chapters have distinctive
enough clinical language that the router identifies them with
near-zero error. Stage-2 resolvers for these chapters will receive
clean, correctly-routed training signal.

**Good routing (92–97%):**
B, D, G, J, K, L, R — acceptable routing reliability. A small number
of records will be misrouted but Stage-2 performance will be
predominantly limited by within-chapter discrimination difficulty
rather than routing errors.

**Problem chapters:**
- **T (70.0%)** — the lowest routing accuracy. T-codes cover
  poisoning, adverse effects, and external causes — clinically
  similar language to S-codes (injuries). With only 10 test records,
  3 misrouted cases dominate the percentage. The T-chapter Stage-2
  resolver will be extremely low-resource (15 classes, ~60 training
  records) and its end-to-end contribution will be limited.
- **Z (90.6%)** — the largest chapter (263 classes, 138 test records)
  and the most misrouted in absolute terms (13 records). Z-codes cover
  administrative and contextual health encounters with highly similar
  language across subclasses — consistent with the E-002 finding that
  Z was the worst-performing chapter at 48.9% flat accuracy.
- **S (91.8%)** — injury codes, semantically adjacent to T-codes.
  4 misrouted test records.

---

### The Routing Error Budget

With 95.3% test routing accuracy, the end-to-end pipeline has a hard
ceiling of 95.3% — any record misrouted by Stage-1 cannot be recovered
by Stage-2 regardless of its within-chapter accuracy.

**End-to-end accuracy estimate:**
If Stage-2 achieves E-002-equivalent within-chapter accuracy (~55%),
the pipeline end-to-end accuracy would be approximately:

> 0.953 × 0.55 ≈ **52.4%** — already exceeding E-002's 46.9%

If Stage-2 outperforms E-002 within chapters (likely, given reduced
label space per resolver), end-to-end accuracy should push higher.

---

### Skip Chapter Decision

Three chapters will not receive dedicated Stage-2 resolvers:

| Chapter | Records | Classes | Reason |
|---|---|---|---|
| U | 5 | 1 | Single class — always predict it |
| P | 25 | 5 | Too few records for reliable resolver |
| Q | 20 | 4 | Too few records for reliable resolver |

For U, P, and Q the Stage-1 routing prediction is used directly
as the ICD-10 prediction — the most frequent within-chapter code
is assigned as the final prediction for any record routed to these
chapters.

---

### Stage-2 Priority Order

Chapters ordered by expected difficulty and clinical impact:

| Priority | Chapter | Routing accuracy | Classes | Strategy |
|---|---|---|---|---|
| 1 | Z | 90.6% | 263 | Full resolver — largest chapter, hardest |
| 2 | R | 94.0% | 196 | Full resolver — symptom ambiguity |
| 3 | M | 98.1% | 222 | Full resolver — largest by records |
| 4 | I | 97.3% | 131 | Full resolver |
| 5 | K | 95.7% | 116 | Full resolver |
| 6–19 | Remaining | varies | varies | Full resolvers in descending size |

> **Stage-1 Status: Complete and locked.**
> Routing accuracy: 95.3% test / 95.5% val.
> Ready for Stage-2 within-chapter resolver training.

</div>

In [ ]:
# Clear old Stage-2 directory before clean re-run
import shutil
shutil.rmtree(STAGE2_DIR, ignore_errors=True)
STAGE2_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Stage-2 directory cleared: {STAGE2_DIR}")
print(f"✅ Ready for clean Phase 4c run")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 4a: Stage-2 Data Preparation

Filters the train/val/test splits by chapter, builds per-chapter label
encoders, and tokenises all 19 trainable chapter datasets. Each chapter
gets its own `DatasetDict` containing only records from that chapter,
with ICD-10 labels re-encoded from 0 to n_classes-1 within the chapter.

### Skip Chapters

Three chapters receive fallback predictions rather than dedicated resolvers:

| Chapter | Records | Classes | Fallback |
|---|---|---|---|
| U | 5 | 1 | Always predict the single U-code |
| P | 25 | 5 | Most frequent within-chapter code |
| Q | 20 | 4 | Most frequent within-chapter code |

### Val/Test Coverage Note

Records in val and test with ICD-10 codes not seen in the chapter
training set are dropped from the chapter-level evaluation. This is
expected — the same coverage limitation as E-002 applies within each
chapter resolver.

</div>

In [ ]:
# ==============================================================================
# PHASE 4a: STAGE-2 DATA PREPARATION — WITHIN-CHAPTER SUBSETS
# ==============================================================================
# Purpose: For each trainable chapter, filter the train/val/test splits to
# records belonging to that chapter, build per-chapter label encoders, and
# prepare the DatasetDicts that each Stage-2 resolver will train on.
#
# Skip chapters: U (1 class), P (5 classes), Q (4 classes)
# These are too small for reliable resolver training. The Stage-1 prediction
# is used directly as the ICD-10 prediction for records routed to these chapters.
#
# All 19 trainable chapters are prepared here. Training happens in Phase 4b.
# ==============================================================================

import json
import numpy as np
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from transformers import AutoTokenizer

# ------------------------------------------------------------------------------
# 1. DEFINE SKIP AND TRAIN CHAPTERS
# ------------------------------------------------------------------------------
SKIP_CHAPTERS = {"U", "P", "Q"}

# Most frequent ICD-10 code per skip chapter — used as fallback prediction
skip_chapter_defaults = {}
for ch in SKIP_CHAPTERS:
    ch_train = train_df[train_df["chapter_label"] == ch]
    if len(ch_train) > 0:
        most_frequent = ch_train["standard_icd10"].value_counts().index[0]
        skip_chapter_defaults[ch] = most_frequent
    else:
        skip_chapter_defaults[ch] = None

print(f"⏭️  Skip chapters (fallback predictions):")
for ch, code in skip_chapter_defaults.items():
    print(f"   {ch}: → {code}")

TRAIN_CHAPTERS = [ch for ch in chapters if ch not in SKIP_CHAPTERS]
print(f"\n✅ Trainable chapters: {len(TRAIN_CHAPTERS)}")
print(f"   {TRAIN_CHAPTERS}")

# ------------------------------------------------------------------------------
# 2. PREPARE PER-CHAPTER DATASETS
# ------------------------------------------------------------------------------
print(f"\n🔧 Preparing per-chapter datasets...")

# Reuse tokenizer from Stage-1
chapter_datasets   = {}   # chapter → DatasetDict
chapter_label_maps = {}   # chapter → {label2id, id2label, num_labels}

for ch in TRAIN_CHAPTERS:
    # Filter to this chapter
    ch_train = train_df[train_df["chapter_label"] == ch].copy()
    ch_val   = val_df[val_df["chapter_label"] == ch].copy()
    ch_test  = test_df[test_df["chapter_label"] == ch].copy()

    if len(ch_train) == 0:
        print(f"   ⚠️  {ch}: no training records — skipping")
        continue

    # Build chapter-specific label encoder
    ch_labels    = sorted(ch_train["standard_icd10"].unique().tolist())
    ch_label2id  = {label: i for i, label in enumerate(ch_labels)}
    ch_id2label  = {i: label for label, i in ch_label2id.items()}
    ch_num_labels = len(ch_label2id)

    # Encode labels
    ch_train = ch_train.copy()
    ch_val   = ch_val.copy()
    ch_test  = ch_test.copy()

    ch_train["ch_label_id"] = ch_train["standard_icd10"].map(ch_label2id)
    ch_val["ch_label_id"]   = ch_val["standard_icd10"].map(ch_label2id)
    ch_test["ch_label_id"]  = ch_test["standard_icd10"].map(ch_label2id)

    # Drop records with unseen labels in val/test
    ch_val  = ch_val.dropna(subset=["ch_label_id"])
    ch_test = ch_test.dropna(subset=["ch_label_id"])

    ch_train["ch_label_id"] = ch_train["ch_label_id"].astype(int)
    ch_val["ch_label_id"]   = ch_val["ch_label_id"].astype(int)
    ch_test["ch_label_id"]  = ch_test["ch_label_id"].astype(int)

    # Build DatasetDict
    ch_features = Features({
        'text':  Value('string'),
        'label': ClassLabel(names=ch_labels)
    })

    datasets_dict = {"train": Dataset.from_dict(
        {"text": ch_train["apso_note"].tolist(),
         "label": ch_train["ch_label_id"].tolist()},
        features=ch_features
    )}

    if len(ch_val) > 0:
        datasets_dict["val"] = Dataset.from_dict(
            {"text": ch_val["apso_note"].tolist(),
             "label": ch_val["ch_label_id"].tolist()},
            features=ch_features
        )

    if len(ch_test) > 0:
        datasets_dict["test"] = Dataset.from_dict(
            {"text": ch_test["apso_note"].tolist(),
             "label": ch_test["ch_label_id"].tolist()},
            features=ch_features
        )

    chapter_datasets[ch]   = DatasetDict(datasets_dict)
    chapter_label_maps[ch] = {
        "label2id":   ch_label2id,
        "id2label":   {str(k): v for k, v in ch_id2label.items()},
        "num_labels": ch_num_labels,
        "chapter":    ch,
    }

    val_size  = len(ch_val)
    test_size = len(ch_test)
    print(f"   {ch:4s}: {len(ch_train):>4,} train | {val_size:>3,} val | "
          f"{test_size:>3,} test | {ch_num_labels:>4,} classes")

# ------------------------------------------------------------------------------
# 3. TOKENISE ALL CHAPTER DATASETS
# ------------------------------------------------------------------------------
print(f"\n🔄 Tokenising all chapter datasets...")

def preprocess_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg["max_length"]
    )

chapter_tokenized = {}

for ch in TRAIN_CHAPTERS:
    if ch not in chapter_datasets:
        continue

    ch_tok = chapter_datasets[ch].map(
        preprocess_fn,
        batched=True,
        remove_columns=["text"]
    )
    ch_tok.set_format("torch")
    chapter_tokenized[ch] = ch_tok

print(f"   ✅ Tokenised {len(chapter_tokenized)} chapter datasets")

# ------------------------------------------------------------------------------
# 4. SAVE CHAPTER LABEL MAPS
# ------------------------------------------------------------------------------
STAGE2_LABEL_MAP_PATH = STAGE2_DIR / "chapter_label_maps.json"
with open(STAGE2_LABEL_MAP_PATH, "w") as f:
    json.dump(chapter_label_maps, f, indent=4)

print(f"\n   ✅ Chapter label maps saved: {STAGE2_LABEL_MAP_PATH.name}")

# ------------------------------------------------------------------------------
# 5. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 4a: Stage-2 Data Preparation",
    action="chapter_datasets_prepared",
    details={
        "trainable_chapters": TRAIN_CHAPTERS,
        "skip_chapters":      list(SKIP_CHAPTERS),
        "skip_defaults":      skip_chapter_defaults,
        "total_chapters":     len(TRAIN_CHAPTERS),
    },
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 4a complete: {len(chapter_tokenized)} chapter datasets ready for Stage-2 training")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 4b: Stage-2 Trainer Configuration

Defines the `train_chapter_resolver()` function used by the Phase 4c
training loop. Each chapter resolver is a **fresh Bio_ClinicalBERT**
— not initialised from E-001.

### Why Fresh Bio_ClinicalBERT for Stage-2?

Stage-1 benefits from E-001 initialisation because E-001 already learned
chapter-level groupings. Stage-2 needs to learn fine-grained within-chapter
code distinctions — a different capability that E-001's ICD-3 training
does not provide. A fresh model with no ICD preconceptions is the correct
starting point for within-chapter discrimination.

### Training Configuration

- **Epochs:** 20 — consistent with E-002
- **Primary metric:** Macro F1 — equal weight to all within-chapter codes
- **`save_total_limit = 2`** — reduced from 3 to manage disk across 19 resolvers
- **`log_level = "error"`** — suppresses per-epoch table output to prevent
  Jupyter's 500-output truncation across 19 × 20 epoch training runs
- **TensorBoard:** All chapter runs log to a shared `stage2/tensorboard/`
  directory — filter by chapter name in the TensorBoard runs panel

</div>

In [ ]:
# ==============================================================================
# PHASE 4b: STAGE-2 TRAINER CONFIGURATION
# ==============================================================================

import os
import io
import contextlib
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback
)
from src.evaluation import hf_compute_metrics

STAGE2_TB_DIR = STAGE2_DIR / "tensorboard"
STAGE2_TB_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TENSORBOARD_LOGGING_DIR"] = str(STAGE2_TB_DIR)

class SilentCallback(TrainerCallback):
    """Suppresses all Trainer output during training."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        pass
    def on_epoch_end(self, args, state, control, **kwargs):
        pass
    def on_evaluate(self, args, state, control, **kwargs):
        pass

def train_chapter_resolver(chapter: str) -> dict:
    """
    Trains a within-chapter ICD-10 resolver for a single chapter.
    Prints one summary line per chapter — all other output suppressed
    via contextlib redirect and SilentCallback.
    """
    if chapter not in chapter_tokenized:
        print(f"   ⚠️  {chapter}: no tokenised data — skipping")
        return {}

    ch_tok        = chapter_tokenized[chapter]
    ch_label_map  = chapter_label_maps[chapter]
    ch_num_labels = ch_label_map["num_labels"]
    ch_label2id   = ch_label_map["label2id"]
    ch_id2label   = {int(k): v for k, v in ch_label_map["id2label"].items()}

    # ------------------------------------------------------------------
    # 1. FRESH MODEL — suppress load report output
    # ------------------------------------------------------------------
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_model = AutoModelForSequenceClassification.from_pretrained(
            cfg["model_name"],
            num_labels=ch_num_labels,
            id2label=ch_id2label,
            label2id=ch_label2id,
            ignore_mismatched_sizes=True,
            cache_dir=str(HF_CACHE_DIR)
        )
    ch_model.to(device)

    assert ch_model.num_labels == ch_num_labels, \
        f"❌ Head mismatch: {ch_model.num_labels} vs {ch_num_labels}"

    # ------------------------------------------------------------------
    # 2. TRAINING ARGUMENTS
    # ------------------------------------------------------------------
    ch_checkpoint_dir = STAGE2_DIR / chapter / "checkpoints"
    ch_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    ch_total_steps  = (
        len(ch_tok["train"]) // cfg["stage2_batch_size"]
    ) * cfg["stage2_num_epochs"]
    ch_warmup_steps = max(1, int(cfg["warmup_ratio"] * ch_total_steps))

    ch_args = TrainingArguments(
        output_dir              = str(ch_checkpoint_dir),
        eval_strategy           = "epoch" if "val" in ch_tok else "no",
        save_strategy           = "epoch",
        load_best_model_at_end  = "val" in ch_tok,
        metric_for_best_model   = "macro_f1",
        greater_is_better       = True,
        save_total_limit        = 2,
        num_train_epochs            = cfg["stage2_num_epochs"],
        per_device_train_batch_size = cfg["stage2_batch_size"],
        learning_rate               = cfg["stage2_learning_rate"],
        weight_decay                = cfg["weight_decay"],
        warmup_steps                = ch_warmup_steps,
        logging_steps               = ch_total_steps + 1,
        report_to                   = ["tensorboard"],
        seed                        = cfg["seed"],
        fp16                        = False,
        dataloader_pin_memory       = False,
        log_level                   = "error",
        disable_tqdm                = True,
    )

    # ------------------------------------------------------------------
    # 3. TRAINER — with SilentCallback
    # ------------------------------------------------------------------
    ch_trainer = Trainer(
        model            = ch_model,
        args             = ch_args,
        train_dataset    = ch_tok["train"],
        eval_dataset     = ch_tok.get("val", None),
        processing_class = tokenizer,
        data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics  = hf_compute_metrics if "val" in ch_tok else None,
        callbacks        = [SilentCallback()],
    )

    # ------------------------------------------------------------------
    # 4. TRAIN — suppress all stdout/stderr
    # ------------------------------------------------------------------
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_result = ch_trainer.train()

    # ------------------------------------------------------------------
    # 5. EXTRACT BEST VAL METRICS
    # ------------------------------------------------------------------
    ch_val_metrics = {}
    if "val" in ch_tok:
        for log in ch_trainer.state.log_history:
            if "eval_macro_f1" in log:
                ch_val_metrics = log

    # ------------------------------------------------------------------
    # 6. SAVE MODEL AND LABEL MAP — suppress shard output
    # ------------------------------------------------------------------
    ch_model_dir = STAGE2_DIR / chapter / "model"
    ch_model_dir.mkdir(parents=True, exist_ok=True)

    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_trainer.save_model(str(ch_model_dir))
        tokenizer.save_pretrained(str(ch_model_dir))

    import json
    with open(ch_model_dir / "label_map.json", "w") as f:
        json.dump(ch_label_map, f, indent=4)

    # ------------------------------------------------------------------
    # 7. ONE SUMMARY LINE PER CHAPTER — the only output
    # ------------------------------------------------------------------
    val_f1   = ch_val_metrics.get("eval_macro_f1", 0)
    val_acc  = ch_val_metrics.get("eval_accuracy", 0)
    val_top5 = ch_val_metrics.get("eval_top_5_accuracy", 0)

    print(f"   ✅ {chapter:4s} | {ch_num_labels:4d} classes | "
          f"loss {ch_result.training_loss:.3f} | "
          f"F1 {val_f1:.3f} | acc {val_acc:.3f} | top5 {val_top5:.3f}")

    return {
        "chapter":      chapter,
        "num_labels":   ch_num_labels,
        "train_loss":   ch_result.training_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc,
        "val_top5":     val_top5,
        "model_path":   str(ch_model_dir),
    }

# ------------------------------------------------------------------
# MONITORING
# ------------------------------------------------------------------
print(f"✅ Stage-2 trainer function defined")
print(f"   19 chapter resolvers will be trained")
print(f"   Each resolver: {cfg['stage2_num_epochs']} epochs, "
      f"lr={cfg['stage2_learning_rate']}, batch={cfg['stage2_batch_size']}")
print(f"   Output: one summary line per chapter")
print(f"   Suppression: contextlib redirect + SilentCallback")

print(f"\n{'='*70}")
print(f"📈 TENSORBOARD (Stage-2 — all chapters):")
print(f"   tensorboard --logdir '{STAGE2_TB_DIR}' --port 6006")
print(f"\n📊 MLFLOW UI:")
print(f"   mlflow ui --backend-store-uri sqlite:///{PROJECT_ROOT}/mlflow.db --port 5001")
print(f"{'='*70}")
print(f"\n✅ Phase 4b complete — run Phase 4c Cell 1 to start Stage-2 training")

# Test

In [ ]:
# ==============================================================================
# PHASE 4b TEST: VERIFY OUTPUT SUPPRESSION (single chapter)
# ==============================================================================
print("🧪 Testing output suppression with Chapter A (smallest chapter)...")
print("   If working correctly you will see exactly ONE summary line below:")
print()
test_result = train_chapter_resolver("A")
print()
print(f"✅ Test complete — output suppression {'WORKING ✅' if test_result else 'FAILED ❌'}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🚀 Phase 4c: Stage-2 Training Ignition — All Chapter Resolvers

Executes the Stage-2 training loop across all 19 trainable chapters in
priority order. Each resolver is trained, evaluated, and saved immediately
before the next chapter begins.

**Priority order:** Chapters with the lowest E-002 flat accuracy and
lowest Stage-1 routing reliability are trained first — Z, R, T, S —
so the most impactful resolvers are available earliest.

**Expected runtime:** approximately 20–25 minutes per chapter resolver
× 19 chapters ≈ **6–8 hours total.** This cell is designed to run
overnight.

**Fault tolerance:** Each chapter is wrapped in a try/except block.
A single chapter failure does not abort the loop — the failure is
logged and training continues with the next chapter.

**Per-chapter results** are logged to MLflow with `stage2_{chapter}_`
prefixed metric names, enabling side-by-side comparison of all 19
resolvers in the MLflow UI.

</div>

In [ ]:
# ==============================================================================
# PHASE 4c CELL 1: STAGE-2 TRAINING LOOP
# ==============================================================================
import json
from datetime import datetime

CHAPTER_PRIORITY = [
    "Z",  # 90.6% routing, 263 classes — hardest
    "R",  # 94.0% routing, 196 classes
    "T",  # 70.0% routing, 15 classes  — unreliable routing
    "S",  # 91.8% routing, 87 classes
    "B",  # 92.3% routing, 31 classes
    "K",  # 95.7% routing, 116 classes
    "M",  # 98.1% routing, 222 classes — largest
    "I",  # 97.3% routing, 131 classes
    "L",  # 95.3% routing, 76 classes
    "D",  # 94.9% routing, 76 classes
    "E",  # 96.9% routing, 76 classes
    "G",  # 96.0% routing, 68 classes
    "O",  # 100%  routing, 63 classes
    "N",  # 97.6% routing, 105 classes
    "J",  # 96.0% routing, 100 classes
    "F",  # 98.2% routing, 95 classes
    "H",  # 97.7% routing, 83 classes
    "C",  # 97.9% routing, 101 classes
    "A",  # 100%  routing, 12 classes
]

assert set(CHAPTER_PRIORITY) == set(TRAIN_CHAPTERS), \
    f"❌ Chapter mismatch: {set(CHAPTER_PRIORITY) ^ set(TRAIN_CHAPTERS)}"

print(f"🚀 Stage-2 Training Loop — {len(CHAPTER_PRIORITY)} chapter resolvers")
print(f"   Init:     Fresh Bio_ClinicalBERT")
print(f"   Started:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n   {'Ch':4s}  {'Classes':>7s}  {'Loss':>6s}  {'F1':>6s}  "
      f"{'Acc':>6s}  {'Top5':>6s}")
print(f"   {'─'*50}")

stage2_results  = {}
stage2_failures = []

for i, chapter in enumerate(CHAPTER_PRIORITY):
    try:
        result = train_chapter_resolver(chapter)
        if result:
            stage2_results[chapter] = result
    except Exception as e:
        print(f"   ❌ {chapter}: FAILED — {e}")
        stage2_failures.append({"chapter": chapter, "error": str(e)})
        continue

print(f"\n   Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Trained:  {len(stage2_results)}/{len(CHAPTER_PRIORITY)}")
print(f"   Failed:   {len(stage2_failures)}")

# Save immediately — before summary or MLflow cells run
STAGE2_RESULTS_PATH = STAGE2_DIR / "stage2_results.json"
with open(STAGE2_RESULTS_PATH, "w") as f:
    json.dump({
        "results":       stage2_results,
        "failures":      stage2_failures,
        "skip_chapters": skip_chapter_defaults,
    }, f, indent=4)

print(f"\n   ✅ Results saved to: {STAGE2_RESULTS_PATH.name}")
print(f"✅ Cell 1 complete — run Cell 2 for summary table")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 4c Cell 2: Stage-2 Val Results Summary

Reads the saved `stage2_results.json` and prints the per-chapter
validation results table. Running this as a separate cell from the
training loop ensures the summary is always available even if Jupyter
truncates the training cell output.

### Results at a Glance

The weighted average val F1 of 0.089 and val accuracy of 13.3% are
consistent with the E-003 analysis — Stage-2 resolvers trained from
fresh Bio_ClinicalBERT face the same fundamental data constraint as
E-002's flat classifier (~4 training examples per ICD-10 code) but
without the cross-chapter representation learning that the flat model
benefited from.

The strongest resolvers (L: 30.3%, N: 26.2%, S: 26.3%, R: 23.3%)
are consistent across runs, confirming these chapters have more
distinctive within-chapter clinical language.

These results establish the official E-003 Stage-2 baseline. The
E-004a experiment will initialise Stage-2 from the E-002 registry
model rather than fresh Bio_ClinicalBERT — the single architectural
change expected to substantially improve within-chapter accuracy.

</div>

In [ ]:
# ==============================================================================
# PHASE 4c CELL 2: SUMMARY TABLE
# ==============================================================================
import json

with open(STAGE2_DIR / "stage2_results.json") as f:
    saved = json.load(f)

results  = saved["results"]
failures = saved["failures"]

print(f"📊 Stage-2 Val Results Summary:")
print(f"   {'Ch':4s}  {'Classes':>7s}  {'Val F1':>8s}  {'Val Acc':>8s}  {'Top-5':>8s}")
print(f"   {'─'*48}")

weighted_f1  = 0
weighted_acc = 0
total_n      = 0

for ch in CHAPTER_PRIORITY:
    if ch not in results:
        print(f"   {ch:4s}  {'—':>7s}  {'FAILED':>8s}  {'FAILED':>8s}  {'FAILED':>8s}")
        continue
    r    = results[ch]
    f1   = r.get("val_macro_f1", 0)
    acc  = r.get("val_accuracy", 0)
    top5 = r.get("val_top5", 0)
    n    = r.get("num_labels", 0)
    weighted_f1  += f1 * n
    weighted_acc += acc * n
    total_n      += n
    print(f"   {ch:4s}  {r['num_labels']:>7,}  {f1:>8.4f}  {acc:>8.4f}  {top5:>8.4f}")

if total_n > 0:
    print(f"\n   {'WTD':4s}  {'':>7s}  "
          f"{weighted_f1/total_n:>8.4f}  {weighted_acc/total_n:>8.4f}")

if failures:
    print(f"\n⚠️  Failed chapters:")
    for f in failures:
        print(f"   {f['chapter']}: {f['error']}")

print(f"\n   Skip chapters: {saved['skip_chapters']}")
print(f"✅ Cell 2 complete — run Cell 3 for MLflow + audit")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 4c Cell 3: MLflow Logging & Audit Trail

Reads `stage2_results.json` and logs all per-chapter metrics to MLflow
with `stage2_{chapter}_` prefixed names, enabling side-by-side comparison
of all 19 resolvers in the MLflow UI. Also logs weighted averages and
closes the audit trail for the Stage-2 training loop.

Running this as a separate cell from the training loop and summary ensures
MLflow logging completes successfully regardless of whether Jupyter
truncated the training cell output.

</div>

In [ ]:
# ==============================================================================
# PHASE 4c CELL 3: MLFLOW LOGGING + AUDIT TRAIL
# ==============================================================================
import json

with open(STAGE2_DIR / "stage2_results.json") as f:
    saved = json.load(f)

results  = saved["results"]
failures = saved["failures"]

# Per-chapter MLflow metrics
for ch, r in results.items():
    mlflow.log_metrics({
        f"stage2_{ch}_val_f1":       r.get("val_macro_f1", 0),
        f"stage2_{ch}_val_accuracy":  r.get("val_accuracy", 0),
        f"stage2_{ch}_train_loss":    r.get("train_loss", 0),
        f"stage2_{ch}_num_labels":    r.get("num_labels", 0),
    })

# Weighted averages
total_n      = sum(r.get("num_labels", 0) for r in results.values())
weighted_f1  = sum(r.get("val_macro_f1", 0) * r.get("num_labels", 0)
                   for r in results.values())
weighted_acc = sum(r.get("val_accuracy", 0) * r.get("num_labels", 0)
                   for r in results.values())

avg_f1  = weighted_f1  / total_n if total_n > 0 else 0
avg_acc = weighted_acc / total_n if total_n > 0 else 0

mlflow.log_metrics({
    "stage2_chapters_trained": len(results),
    "stage2_chapters_failed":  len(failures),
    "stage2_weighted_val_f1":  avg_f1,
    "stage2_weighted_val_acc": avg_acc,
})

config.log_event(
    phase="Phase 4c: Stage-2 Training Loop",
    action="stage2_training_complete",
    details={
        "chapters_trained": len(results),
        "chapters_failed":  len(failures),
        "weighted_val_f1":  avg_f1,
        "weighted_val_acc": avg_acc,
    },
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"   Chapters trained: {len(results)}/19")
print(f"   Weighted val F1:  {avg_f1:.4f}")
print(f"   Weighted val Acc: {avg_acc:.4f}")
print(f"\n📝 Audit trail updated")
print(f"✅ Phase 4c complete — proceed to Phase 5")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🎯 Phase 5: End-to-End Pipeline Evaluation

Routes every test record through the full two-stage pipeline and measures
end-to-end ICD-10 prediction accuracy against E-002's flat baseline.

### Pipeline Logic

Each test record passes through two stages:

1. **Stage-1 chapter router** — predicts which of 22 ICD-10 chapters
   the record belongs to (95.3% routing accuracy on test set)
2. **Stage-2 within-chapter resolver** — predicts the specific ICD-10
   code within the routed chapter

Skip chapters (U, P, Q) bypass Stage-2 and receive fallback predictions
— the most frequent within-chapter ICD-10 code from the training set.

### What This Measures

The end-to-end evaluation answers the core E-003 hypothesis:

> Does routing through a high-accuracy chapter classifier before
> resolving within-chapter codes outperform E-002's flat 46.9%
> accuracy and 0.352 Macro F1?

The per-record routing log also decomposes errors into two categories:
Stage-1 routing errors (irrecoverable) and Stage-2 resolution errors
(within-chapter discrimination failures), giving a clear picture of
where the pipeline fails and what would most improve end-to-end accuracy.

### Baseline to Beat

| Metric | E-002 flat |
|---|---|
| Accuracy | 46.9% |
| Macro F1 | 0.352 |

</div>

In [ ]:
# ==============================================================================
# PHASE 5: END-TO-END PIPELINE EVALUATION (E-003)
# ==============================================================================
# Purpose: Route every test record through Stage-1 (chapter router) then
# Stage-2 (within-chapter resolver) and measure end-to-end ICD-10 accuracy.
#
# Pipeline logic:
#   1. Stage-1 predicts the ICD-10 chapter for each record
#   2. The record is routed to the corresponding Stage-2 resolver
#   3. Stage-2 predicts the specific ICD-10 code within that chapter
#   4. Skip chapters (U, P, Q) use fallback predictions directly
#
# Evaluation compares end-to-end predictions against true ICD-10 labels
# using the same metrics as E-001 and E-002 for direct comparison.
# ==============================================================================

import json
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from collections import defaultdict

# ------------------------------------------------------------------------------
# 1. LOAD ALL STAGE-2 RESOLVER MODELS INTO MEMORY
# ------------------------------------------------------------------------------
print(f"📥 Loading Stage-2 resolver models...")

stage2_models    = {}
stage2_tokenizers = {}
stage2_label_maps = {}

for ch in CHAPTER_PRIORITY:
    ch_model_dir = STAGE2_DIR / ch / "model"
    if not ch_model_dir.exists():
        print(f"   ⚠️  Chapter {ch}: model not found — skipping")
        continue

    with open(ch_model_dir / "label_map.json") as f:
        lmap = json.load(f)

    ch_model = AutoModelForSequenceClassification.from_pretrained(
        str(ch_model_dir)
    )
    ch_model.to(device)
    ch_model.eval()

    ch_tokenizer = AutoTokenizer.from_pretrained(str(ch_model_dir))

    stage2_models[ch]     = ch_model
    stage2_tokenizers[ch] = ch_tokenizer
    stage2_label_maps[ch] = lmap

    print(f"   ✅ {ch}: {lmap['num_labels']} classes loaded")

print(f"\n   ✅ {len(stage2_models)}/19 resolvers loaded into memory")

# ------------------------------------------------------------------------------
# 2. STAGE-1 PREDICTIONS ON TEST SET
# ------------------------------------------------------------------------------
print(f"\n🧭 Running Stage-1 chapter routing on test set...")

stage1_model.eval()
stage1_test_preds = []
stage1_test_probs = []

with torch.no_grad():
    for i in range(0, len(stage1_tokenized["test"]), 32):
        batch = {
            k: stage1_tokenized["test"][i:i+32][k].to(device)
            for k in ["input_ids", "attention_mask", "token_type_ids"]
        }
        outputs = stage1_model(**batch)
        probs   = torch.softmax(outputs.logits, dim=-1)
        preds   = torch.argmax(probs, dim=-1)
        stage1_test_preds.extend(preds.cpu().numpy().tolist())
        stage1_test_probs.extend(probs.cpu().numpy().tolist())

stage1_test_preds = np.array(stage1_test_preds)
true_chapter_ids  = np.array(stage1_tokenized["test"]["label"])
true_icd10_codes  = test_df["standard_icd10"].tolist()

stage1_routing_acc = (stage1_test_preds == true_chapter_ids).mean()
print(f"   ✅ Stage-1 routing accuracy: {stage1_routing_acc:.1%}")

# ------------------------------------------------------------------------------
# 3. END-TO-END PIPELINE PREDICTIONS
# ------------------------------------------------------------------------------
print(f"\n🔬 Running end-to-end pipeline predictions...")

pipeline_predictions = []   # final predicted ICD-10 code per record
pipeline_true        = []   # true ICD-10 code per record
routing_log          = []   # per-record routing details

test_texts = test_df["apso_note"].tolist()

for idx, (text, true_code) in enumerate(zip(test_texts, true_icd10_codes)):

    # Stage-1: get predicted chapter
    pred_chapter_id   = int(stage1_test_preds[idx])
    pred_chapter      = id2chapter[pred_chapter_id]
    true_chapter      = true_code[0]

    correctly_routed  = (pred_chapter == true_chapter)

    # Stage-2: predict ICD-10 code within predicted chapter
    if pred_chapter in SKIP_CHAPTERS:
        # Fallback prediction for skip chapters
        pred_icd10 = skip_chapter_defaults.get(pred_chapter, "UNKNOWN")
        stage2_source = "fallback"

    elif pred_chapter not in stage2_models:
        # Resolver not available — use most frequent code in chapter
        ch_train = train_df[train_df["chapter_label"] == pred_chapter]
        pred_icd10 = ch_train["standard_icd10"].value_counts().index[0] \
            if len(ch_train) > 0 else "UNKNOWN"
        stage2_source = "fallback_no_model"

    else:
        # Run Stage-2 resolver
        ch_model     = stage2_models[pred_chapter]
        ch_tokenizer = stage2_tokenizers[pred_chapter]
        ch_lmap      = stage2_label_maps[pred_chapter]
        ch_id2label  = {int(k): v for k, v in ch_lmap["id2label"].items()}

        encoding = ch_tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=cfg["max_length"],
            return_tensors="pt"
        )
        encoding = {k: v.to(device) for k, v in encoding.items()
                    if k in ["input_ids", "attention_mask", "token_type_ids"]}

        with torch.no_grad():
            outputs    = ch_model(**encoding)
            pred_id    = torch.argmax(outputs.logits, dim=-1).item()
            pred_icd10 = ch_id2label.get(pred_id, "UNKNOWN")

        stage2_source = "resolver"

    pipeline_predictions.append(pred_icd10)
    pipeline_true.append(true_code)

    routing_log.append({
        "idx":              idx,
        "true_code":        true_code,
        "true_chapter":     true_chapter,
        "pred_chapter":     pred_chapter,
        "correctly_routed": correctly_routed,
        "pred_icd10":       pred_icd10,
        "correct":          pred_icd10 == true_code,
        "stage2_source":    stage2_source,
    })

    if (idx + 1) % 100 == 0:
        correct_so_far = sum(r["correct"] for r in routing_log)
        print(f"   [{idx+1}/{len(test_texts)}] Running accuracy: "
              f"{correct_so_far/(idx+1):.1%}")

# ------------------------------------------------------------------------------
# 4. COMPUTE END-TO-END METRICS
# ------------------------------------------------------------------------------
print(f"\n📊 Computing end-to-end metrics...")

from sklearn.metrics import accuracy_score, f1_score

y_true_codes = pipeline_true
y_pred_codes = pipeline_predictions

# Top-1 accuracy and Macro F1
e2e_accuracy  = accuracy_score(y_true_codes, y_pred_codes)
e2e_macro_f1  = f1_score(
    y_true_codes, y_pred_codes,
    average="macro", zero_division=0
)

# Error analysis
n_correct         = sum(r["correct"] for r in routing_log)
n_routing_correct = sum(r["correctly_routed"] for r in routing_log)
n_routing_errors  = sum(not r["correctly_routed"] for r in routing_log)
n_stage2_correct  = sum(
    r["correct"] for r in routing_log if r["correctly_routed"]
)

# Within-chapter accuracy (on correctly routed records only)
within_chapter_acc = n_stage2_correct / n_routing_correct \
    if n_routing_correct > 0 else 0

print(f"\n{'='*60}")
print(f"  E-003 END-TO-END PIPELINE RESULTS")
print(f"{'='*60}")
print(f"  Test records:            {len(routing_log):,}")
print(f"{'─'*60}")
print(f"  STAGE-1 ROUTING:")
print(f"    Correctly routed:      {n_routing_correct:,} ({n_routing_correct/len(routing_log):.1%})")
print(f"    Misrouted:             {n_routing_errors:,} ({n_routing_errors/len(routing_log):.1%})")
print(f"{'─'*60}")
print(f"  STAGE-2 WITHIN-CHAPTER (on correctly routed):")
print(f"    Correct predictions:   {n_stage2_correct:,} ({within_chapter_acc:.1%})")
print(f"{'─'*60}")
print(f"  END-TO-END:")
print(f"    Accuracy:              {e2e_accuracy:.1%}")
print(f"    Macro F1:              {e2e_macro_f1:.4f}")
print(f"{'─'*60}")
print(f"  E-002 BASELINE:")
print(f"    Accuracy:              46.9%")
print(f"    Macro F1:              0.352")
print(f"{'─'*60}")
delta_acc = e2e_accuracy - 0.469
delta_f1  = e2e_macro_f1 - 0.352
print(f"  DELTA vs E-002:")
print(f"    Accuracy:              {delta_acc:+.1%}")
print(f"    Macro F1:              {delta_f1:+.4f}")
print(f"{'='*60}")

# ------------------------------------------------------------------------------
# 5. PER-CHAPTER END-TO-END BREAKDOWN
# ------------------------------------------------------------------------------
print(f"\n📊 Per-chapter end-to-end accuracy:")
print(f"   {'Ch':4s}  {'True':>6s}  {'Routed':>7s}  {'Correct':>8s}  {'E2E Acc':>8s}")
print(f"   {'─'*45}")

chapter_e2e = defaultdict(lambda: {"total": 0, "routed": 0, "correct": 0})
for r in routing_log:
    ch = r["true_chapter"]
    chapter_e2e[ch]["total"]   += 1
    chapter_e2e[ch]["routed"]  += int(r["correctly_routed"])
    chapter_e2e[ch]["correct"] += int(r["correct"])

for ch in sorted(chapter_e2e.keys()):
    stats = chapter_e2e[ch]
    e2e_acc = stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"   {ch:4s}  {stats['total']:>6,}  {stats['routed']:>7,}"
          f"  {stats['correct']:>8,}  {e2e_acc:>8.1%}")

# ------------------------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------------------------
e2e_summary = {
    "e2e_accuracy":       e2e_accuracy,
    "e2e_macro_f1":       e2e_macro_f1,
    "stage1_routing_acc": float(stage1_routing_acc),
    "within_chapter_acc": within_chapter_acc,
    "n_correct":          n_correct,
    "n_routing_correct":  n_routing_correct,
    "n_routing_errors":   n_routing_errors,
    "n_stage2_correct":   n_stage2_correct,
    "e002_accuracy":      0.469,
    "e002_macro_f1":      0.352,
    "delta_accuracy":     delta_acc,
    "delta_macro_f1":     delta_f1,
}

with open(EXP_DIR / "e2e_results.json", "w") as f:
    json.dump(e2e_summary, f, indent=4)

mlflow.log_metrics({
    "e2e_accuracy":       e2e_accuracy,
    "e2e_macro_f1":       e2e_macro_f1,
    "stage1_routing_acc": float(stage1_routing_acc),
    "within_chapter_acc": within_chapter_acc,
})

config.log_event(
    phase="Phase 5: End-to-End Evaluation",
    action="e2e_evaluation_complete",
    details=e2e_summary,
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 5 complete")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 E-003 End-to-End Results: Interpretation

### Official E-003 Results

| Metric | E-001 (ICD-3) | E-002 (ICD-10 flat) | E-003 (Hierarchical) |
|---|---|---|---|
| Accuracy | 82.7% | 46.9% | 10.6% |
| Macro F1 | 0.760 | 0.352 | 0.070 |
| Approach | 675 classes | 1,926 classes flat | 22 → 1,926 two-stage |

**E-003 did not outperform E-002.** The hierarchical pipeline achieved
10.6% end-to-end accuracy versus E-002's 46.9% — a -36.3pp deficit.

---

### Stage-by-Stage Breakdown

| Stage | Metric | Value |
|---|---|---|
| Stage-1 routing | Test accuracy | 95.3% |
| Stage-1 routing | Test Macro F1 | 0.959 |
| Stage-2 resolvers | Within-chapter accuracy | 11.1% |
| End-to-end | Accuracy | 10.6% |
| End-to-end | Macro F1 | 0.070 |

Stage-1 performed exceptionally — 95.3% chapter routing accuracy
substantially exceeded E-002's flat chapter accuracy of 82.9%.
The pipeline failure is entirely attributable to Stage-2.

---

### Why Stage-2 Underperformed

**Training data fragmentation** is the primary cause. The flat E-002
classifier trained on all 7,728 records simultaneously, learning
cross-chapter representations that implicitly aided within-chapter
discrimination. Each Stage-2 resolver trains in isolation on only
its chapter's records — Z-chapter gets 1,056, M-chapter gets 888,
A-chapter gets only 48. The resolvers have significantly less
training signal than E-002's flat model.

**No pre-trained ICD representations.** Stage-2 uses fresh
Bio_ClinicalBERT with no ICD knowledge. E-002's flat model built
rich ICD-10 representations over 20 epochs on 7,728 records. Each
Stage-2 resolver must learn these representations from scratch on
a fraction of the data.

**The correct initialisation for Stage-2 is E-002, not fresh
Bio_ClinicalBERT.** E-002 already has strong within-chapter
representations. Fine-tuning E-002 on chapter-filtered data would
combine the broad representation learning of the flat approach with
the focused within-chapter objective of the hierarchical approach.
This is the key architectural fix for E-004.

---

### Per-Chapter Performance

The per-chapter breakdown reveals that some resolvers achieved
meaningful accuracy despite the overall pipeline underperformance:

**Strongest resolvers:**
N-codes (26.8%), R-codes (21.7%), S-codes (20.4%), Z-codes (14.5%)
— these chapters have more distinctive within-chapter terminology
and larger val/test sets for reliable measurement.

**Zero-accuracy chapters:**
B, H, Q — these chapters have extremely small test sets (1-13
records) making the 0% figures unreliable rather than indicative
of complete model failure.

---

### What This Confirms

The hierarchical approach is architecturally sound — Stage-1 proved
that a dedicated chapter router substantially outperforms implicit
chapter-level accuracy from a flat classifier. The failure is in the
Stage-2 implementation, not the concept.

Three specific findings inform the next experiment:

1. **Stage-1 from E-001 works.** The E-001 initialisation gave the
   chapter router a decisive head start. This approach should be
   preserved in future hierarchical experiments.

2. **Stage-2 should be initialised from E-002, not fresh
   Bio_ClinicalBERT.** The flat ICD-10 model already has the
   within-chapter representations Stage-2 needs. Fine-tuning rather
   than training from scratch is the correct approach.

3. **The data fragmentation problem is fundamental to MedSynth.**
   With only ~5 records per ICD-10 code, any approach that splits
   the training data by chapter will face severe data constraints.
   Real clinical datasets with natural class imbalance would give
   large chapters (M, Z, I) enough data to train effective resolvers.

---

### Recommendations for E-004

| Experiment | Change | Expected Impact |
|---|---|---|
| E-004a | Initialise Stage-2 from E-002 registry model | Primary fix — leverages existing ICD-10 representations |
| E-004b | Train Stage-2 with full dataset, chapter-masked loss | Prevents data fragmentation |
| E-004c | Top-k routing (k=3) — let Stage-2 see top-3 chapters | Reduces routing error propagation |
| E-004d | Apply to real clinical dataset | Resolves MedSynth data constraint |

> **E-003 Status: Complete.** End-to-end accuracy = 10.6%,
> Macro F1 = 0.070. Stage-1 chapter router (95.3% accuracy)
> is a strong result preserved for future experiments.
> Stage-2 requires E-002 initialisation — scheduled for E-004a.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🏆 Phase 6: Registry Promotion (E-003)

Saves the Stage-1 chapter router and experiment metadata to the permanent
registry. Stage-2 resolvers are already saved to `stage2/{chapter}/model/`
during training — this phase promotes the Stage-1 model and official
results for long-term reference.

### Registry Contents

| Artifact | Location |
|---|---|
| Stage-1 router weights | `registry/E-003_Hierarchical_ICD10/stage1/` |
| Chapter mapping | `registry/E-003_Hierarchical_ICD10/stage1/chapter_mapping.json` |
| Stage-2 resolvers | `stage2/{chapter}/model/` (19 resolvers) |
| Final metrics | `registry/E-003_Hierarchical_ICD10/final_metrics.json` |
| E2E results | `registry/E-003_Hierarchical_ICD10/e2e_results.json` |
| Experiment config | `registry/E-003_Hierarchical_ICD10/experiment_config.json` |

### What to Preserve for E-004a

The Stage-1 chapter router is the most valuable E-003 artifact. E-004a
will reuse it directly — no Stage-1 retraining required. Only Stage-2
needs to change: initialise from the E-002 registry model instead of
fresh Bio_ClinicalBERT.

</div>

In [ ]:
# ==============================================================================
# PHASE 6: REGISTRY PROMOTION (E-003)
# ==============================================================================
# Saves the Stage-1 chapter router and E-003 experiment summary to the
# permanent registry. Stage-2 resolvers are already saved to stage2/{ch}/model/
# during training — this phase promotes the Stage-1 model and experiment
# metadata to the registry for long-term reference.
# ==============================================================================

import json
import shutil
from datetime import datetime

print(f"🏆 Promoting E-003 artifacts to registry...")

# ------------------------------------------------------------------------------
# 1. SAVE STAGE-1 ROUTER TO REGISTRY
# ------------------------------------------------------------------------------
REGISTRY_STAGE1_DIR = REGISTRY_DIR / "stage1"
REGISTRY_STAGE1_DIR.mkdir(parents=True, exist_ok=True)

stage1_trainer.save_model(str(REGISTRY_STAGE1_DIR))
tokenizer.save_pretrained(str(REGISTRY_STAGE1_DIR))

shutil.copy(
    STAGE1_DIR / "chapter_mapping.json",
    REGISTRY_STAGE1_DIR / "chapter_mapping.json"
)
shutil.copy(
    STAGE1_DIR / "chapter_test_stats.json",
    REGISTRY_STAGE1_DIR / "chapter_test_stats.json"
)

print(f"   ✅ Stage-1 router saved to registry")

# ------------------------------------------------------------------------------
# 2. SAVE FINAL METRICS
# ------------------------------------------------------------------------------
final_metrics = {
    "experiment_id":        cfg["experiment_id"],
    "experiment_name":      cfg["experiment_name"],
    "timestamp":            datetime.now().isoformat(),

    "stage1": {
        "init_model":       cfg["stage1_init_model"],
        "num_classes":      num_chapters,
        "num_epochs":       cfg["stage1_num_epochs"],
        "val_accuracy":     0.9555,
        "val_macro_f1":     0.9677,
        "val_top5":         0.9855,
        "test_accuracy":    0.9534,
        "test_macro_f1":    0.9589,
        "test_top5":        0.9834,
    },

    "stage2": {
        "num_resolvers":    len(stage2_results),
        "skip_chapters":    list(SKIP_CHAPTERS),
        "weighted_val_f1":  0.0920,
        "weighted_val_acc": 0.1360,
        "init_model":       "fresh Bio_ClinicalBERT",
    },

    "end_to_end": {
        "test_accuracy":    0.1055,
        "test_macro_f1":    0.0702,
        "stage1_routing":   0.9534,
        "within_chapter":   0.1107,
    },

    "baseline_e002": {
        "test_accuracy":    0.469,
        "test_macro_f1":    0.352,
    },

    "delta_vs_e002": {
        "accuracy":         0.1055 - 0.469,
        "macro_f1":         0.0702 - 0.352,
    },

    "key_finding": (
        "Stage-1 chapter router (95.3% accuracy) substantially exceeded "
        "E-002 flat chapter accuracy (82.9%). Stage-2 underperformed due "
        "to training data fragmentation and fresh initialisation. "
        "Fix: initialise Stage-2 from E-002 registry model in E-004a."
    )
}

with open(REGISTRY_DIR / "final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=4)

print(f"   ✅ Final metrics saved to registry")

# ------------------------------------------------------------------------------
# 3. SAVE EXPERIMENT CONFIG
# ------------------------------------------------------------------------------
with open(REGISTRY_DIR / "experiment_config.json", "w") as f:
    json.dump({k: v for k, v in cfg.items()
               if not isinstance(v, list)}, f, indent=4)

print(f"   ✅ Experiment config saved to registry")

# ------------------------------------------------------------------------------
# 4. SAVE E2E RESULTS
# ------------------------------------------------------------------------------
shutil.copy(
    EXP_DIR / "e2e_results.json",
    REGISTRY_DIR / "e2e_results.json"
)

print(f"   ✅ E2E results saved to registry")

# ------------------------------------------------------------------------------
# 5. CLOSE MLFLOW RUN
# ------------------------------------------------------------------------------
if mlflow.active_run():
    mlflow.end_run()
    print(f"   ✅ MLflow run closed")

# ------------------------------------------------------------------------------
# 6. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 6: Registry Promotion",
    action="e003_registry_promotion_complete",
    details={
        "registry_path":    str(REGISTRY_DIR),
        "stage1_promoted":  True,
        "metrics_saved":    True,
        "mlflow_closed":    True,
    },
    notebook="04-Model_Hierarchical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"\n{'='*60}")
print(f"  E-003 REGISTRY SUMMARY")
print(f"{'='*60}")
print(f"  Registry:       {REGISTRY_DIR.name}/")
print(f"  Stage-1 model:  registry/stage1/")
print(f"  Stage-2 models: stage2/{{chapter}}/model/")
print(f"  Metrics:        final_metrics.json")
print(f"  E2E results:    e2e_results.json")
print(f"{'='*60}")
print(f"  Stage-1 test accuracy:  95.3%")
print(f"  Stage-1 test Macro F1:  0.959")
print(f"  E2E test accuracy:      10.6%")
print(f"  E2E test Macro F1:      0.070")
print(f"  Status: COMPLETE — Stage-2 fix scheduled for E-004a")
print(f"{'='*60}")
print(f"\n✅ Phase 6 complete")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🧪 Experiment Log: Scientific Record (E-003)

### Configuration

| Parameter | Value |
|---|---|
| **Architecture** | Two-stage hierarchical |
| **Stage-1 model** | E-001 registry (Bio_ClinicalBERT, ICD-3 pretrained) |
| **Stage-2 model** | Fresh Bio_ClinicalBERT (19 chapter resolvers) |
| **Input** | `apso_note` — APSO-recomposed, ICD-10-redacted |
| **Records** | 9,660 billable (7,728 train / 966 val / 966 test) |
| **Stage-1 classes** | 22 ICD-10 chapters |
| **Stage-2 classes** | 1,926 ICD-10 codes (split across 19 resolvers) |
| **Stage-1 epochs** | 10 |
| **Stage-2 epochs** | 20 per resolver |
| **Skip chapters** | U, P, Q — fallback predictions |
| **Hardware** | Apple Silicon MPS |
| **Tracking** | MLflow SQLite backend |

---

### Official Results

**Stage-1 Chapter Router:**

| Metric | Val | Test |
|---|---|---|
| Accuracy | 95.5% | 95.3% |
| Macro F1 | 0.968 | 0.959 |
| Top-5 Accuracy | 98.6% | 98.3% |

**Stage-2 Within-Chapter Resolvers (weighted val averages):**

| Metric | Value |
|---|---|
| Weighted val Macro F1 | 0.092 |
| Weighted val Accuracy | 13.6% |
| Within-chapter test accuracy | 11.1% |

**End-to-End Pipeline (test set):**

| Metric | E-001 | E-002 | E-003 | Delta vs E-002 |
|---|---|---|---|---|
| Accuracy | 82.7% | 46.9% | 10.6% | -36.3pp |
| Macro F1 | 0.760 | 0.352 | 0.070 | -0.282 |
| Chapter accuracy | n/a | 82.9% | 95.3% | +12.4pp |

---

### Key Findings

**Stage-1 succeeded.** The E-001-initialised chapter router achieved
95.3% test routing accuracy — a +12.4pp improvement over E-002's
implicit flat chapter accuracy of 82.9%. The dedicated routing
objective with E-001 pretraining is a validated approach.

**Stage-2 failed due to training data fragmentation.** Each of the
19 chapter resolvers trained in isolation on only its chapter's
records — Z-chapter got 1,056 records, M-chapter 888, A-chapter
only 48. The flat E-002 classifier trained on all 7,728 records
simultaneously and learned richer cross-chapter representations
that aided within-chapter discrimination.

**Fresh Bio_ClinicalBERT was the wrong initialisation for Stage-2.**
E-002 already has strong ICD-10 representations from 20 epochs on
7,728 records. Initialising Stage-2 resolvers from E-002 would
provide these representations as a starting point rather than
requiring each resolver to learn them from scratch on a fraction
of the data.

---

### E-004a Fix

The single architectural change required:
```python
# Phase 4b: replace this
cfg["model_name"]  # fresh Bio_ClinicalBERT

# with this
E002_MODEL_PATH = REGISTRY_BASE / "E-002_FullICD10_ClinicalBERT" / "model"
```

Each Stage-2 resolver will load E-002 weights and fine-tune on
chapter-filtered data, combining E-002's broad ICD-10 representations
with chapter-specific training signal.

> **STATUS: COMPLETE.**
> Stage-1 accuracy: 95.3%. E2E accuracy: 10.6%.
> Stage-2 fix (E-002 initialisation) scheduled for E-004a.

</div>